<h1 style="text-align:center; color:blue; font-size:32px;">
  <b>On the top Menu click "Run" and then "Run All Cells"</b>
</h1>


<h2 style="text-align:center; font-size:26px;">
  📺 Watch <a href="https://www.youtube.com/watch?v=lC2y_dAHKvM" target="_blank">Tutorial Video</a>
</h2>


In [ ]:
import warnings
import ee
import numpy as np
import pandas as pd
import math
import os
import io
import base64
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# Install and import required packages
packages_to_install = ['ipyleaflet', 'gspread', 'google-auth']
for package in packages_to_install:
    try:
        if package == 'ipyleaflet':
            import ipyleaflet
        elif package == 'gspread':
            import gspread
        elif package == 'google-auth':
            import google.auth
    except ImportError:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

from ipyleaflet import Map, DrawControl, basemaps, WidgetControl, GeoJSON
import gspread
from google.oauth2.service_account import Credentials
warnings.filterwarnings('ignore')
try:
    import logging as _logging
    _logging.getLogger('lightgbm').setLevel(_logging.ERROR)
except Exception:
    pass

# --- Optional imports for classifying algorithms ---
def _get_xgb():
    try:
        import xgboost as xgb
    except Exception:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
        import xgboost as xgb
    return xgb

def _get_lgb():
    try:
        import lightgbm as lgb
    except Exception:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "lightgbm"])
        import lightgbm as lgb
    return lgb

def _get_nominatim():
    try:
        from geopy.geocoders import Nominatim
    except ImportError:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "geopy"])
        from geopy.geocoders import Nominatim
    return Nominatim

# ===============================================
# PROJECT CONFIGURATION
# ===============================================
PROJECT_ID = "animated-rhythm-449415-u3"

# Google Sheets Configuration
GOOGLE_SHEET_NAME = "AlphaEarth User Interactions"
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1XXhGbSc5_UwXmsuvhkJJnqdrilxgPGvo-t1AYTcP6T0/edit?gid=0#gid=0"

# ===============================================
# SIMPLE EARTH ENGINE AUTHENTICATION
# ===============================================
EE_INITIALIZED = False

print("Authenticating with Google Earth Engine...")
try:
    ee.Initialize()
    print("Earth Engine already initialized!")
    EE_INITIALIZED = True
except:
    try:
        print("Starting Earth Engine authentication...")
        print("Click the link below, sign in, and paste the code when prompted")
        ee.Authenticate()
        ee.Initialize()
        print("Earth Engine authenticated successfully!")
        EE_INITIALIZED = True
    except Exception as e:
        print(f"Earth Engine authentication failed: {str(e)}")
        print("Some features will be limited")
        EE_INITIALIZED = False

print("Welcome to AlphaEarth Land Cover Classifier!")
print("Loading app...")

# ===============================================
# ORIGINAL WORKING FUNCTIONS
# ===============================================

# Color palettes for embedding visualization (up to 10)
EMBEDDING_PALETTES = {
    'embedding_1':  ['#000080', '#0066CC', '#00CCFF', '#66FFCC', '#CCFF66', '#FFCC00', '#FF6600', '#CC0000'],
    'embedding_2':  ['#2D0066', '#6600CC', '#9966FF', '#CC99FF', '#FFCCFF', '#FFCC99', '#FF9966', '#FF6633'],
    'embedding_3':  ['#004D00', '#009900', '#33CC33', '#66FF66', '#99FF99', '#CCFF33', '#FFFF00', '#FFB300'],
    'embedding_4':  ['#003300', '#006600', '#009933', '#00CC66', '#66FF99', '#CCFF99', '#FFFF66', '#FFCC33'],
    'embedding_5':  ['#1A0000', '#660000', '#CC0000', '#FF4444', '#FF9999', '#FFCCCC', '#FFE6E6', '#FFFFFF'],
    'embedding_6':  ['#000033', '#000099', '#0033FF', '#3366FF', '#6699FF', '#99CCFF', '#CCDDFF', '#EEF5FF'],
    'embedding_7':  ['#1A1A00', '#666600', '#999900', '#CCCC00', '#FFFF00', '#FFFF66', '#FFFFAA', '#FFFFDD'],
    'embedding_8':  ['#330033', '#660066', '#990099', '#CC00CC', '#FF00FF', '#FF66FF', '#FFAAFF', '#FFDDFF'],
    'embedding_9':  ['#001A1A', '#003333', '#006666', '#009999', '#00CCCC', '#33FFFF', '#99FFFF', '#CCFFFF'],
    'embedding_10': ['#1A0A00', '#4D1F00', '#803300', '#B34700', '#E65C00', '#FF8533', '#FFAD85', '#FFD9C2'],
}

def sanitize_bbox(bbox):
    """From working code - robust bbox handling"""
    minLon, minLat, maxLon, maxLat = map(float, bbox)

    def clamp(v, lo, hi): return max(lo, min(hi, v))
    minLon = ((minLon + 180) % 360) - 180
    maxLon = ((maxLon + 180) % 360) - 180
    minLat = clamp(minLat, -85, 85)
    maxLat = clamp(maxLat, -85, 85)

    if minLat > maxLat:
        minLat, maxLat = maxLat, minLat

    if abs(maxLon - minLon) < 1e-6:
        maxLon = min(180.0, minLon + 0.01)

    parts = []
    if minLon <= maxLon:
        parts = [[minLon, minLat, maxLon, maxLat]]
    else:
        parts = [
            [minLon, minLat, 180.0, maxLat],
            [-180.0, minLat, maxLon, maxLat]
        ]

    center_lon = math.fsum([p[0] + p[2] for p in parts]) / (2 * len(parts))
    center_lat = (minLat + maxLat) / 2.0
    return {"parts": parts, "center": [center_lat, center_lon]}

def region_from_parts(parts):
    """From working code"""
    rects = [ee.Geometry.Rectangle(p) for p in parts]
    geom = rects[0] if len(rects) == 1 else ee.Geometry.MultiPolygon(rects).dissolve()
    return geom

def load_stack(year=2020):
    """From working code"""
    emb_all = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
               .filterDate(f'{year}-01-01', f'{year+1}-01-01')
               .mosaic())
    bands = emb_all.bandNames()
    emb = emb_all.select(bands)
    emb_mask = emb.select(0).mask()
    esa = ee.Image("ESA/WorldCover/v100/2020").select('Map').rename('label')
    return emb, bands, emb_mask, esa

def add_ee_layer(folium_map, ee_image_object, vis_params, name):
    """From working code - exactly as it was"""
    import folium
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
        name=name,
        overlay=True,
        control=True
    ).add_to(folium_map)

def normalize_embedding(embedding_image, region):
    """From working code - EXACTLY as it was working"""
    stats = embedding_image.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=region,
        scale=1000,
        maxPixels=1e9
    )

    band_name = embedding_image.bandNames().get(0)
    min_val = ee.Number(stats.get(ee.String(band_name).cat('_min'))).max(-10)
    max_val = ee.Number(stats.get(ee.String(band_name).cat('_max'))).min(10)

    normalized = embedding_image.subtract(min_val).divide(max_val.subtract(min_val)).clamp(0, 1)
    return normalized

def top_k_importance(importances, bands, k=5, as_percent=True):
    """From working code"""
    vals = np.asarray(importances, dtype=float).ravel()
    bands = list(bands)
    L = min(vals.size, len(bands))
    vals, bands = vals[:L], bands[:L]
    disp = [f"A{int(b[1:]) + 1:02d}" for b in bands]
    if as_percent:
        total = vals.sum() or 1.0
        vals = 100.0 * vals / total
    order = np.argsort(vals)[::-1][:k]
    return [(disp[i], float(vals[i]), bands[i]) for i in order]

def all_embedding_importance(importances, bands, as_percent=True):
    """Get importance for all 64 embeddings in A01-A64 order"""
    vals = np.asarray(importances, dtype=float).ravel()
    bands = list(bands)
    L = min(vals.size, len(bands))
    vals, bands = vals[:L], bands[:L]

    band_to_importance = {}
    for i, band in enumerate(bands):
        display_name = f"A{int(band[1:]) + 1:02d}"
        band_to_importance[display_name] = vals[i]

    if as_percent:
        total = vals.sum() or 1.0
        for key in band_to_importance:
            band_to_importance[key] = 100.0 * band_to_importance[key] / total

    result = []
    for i in range(1, 65):
        embedding_name = f"A{i:02d}"
        importance = band_to_importance.get(embedding_name, 0.0)
        result.append((embedding_name, float(importance)))

    return result

# ===============================================
# GOOGLE SHEETS DATA LOGGING
# ===============================================

def setup_google_sheets():
    """Setup Google Sheets connection using environment variables for credentials"""
    try:
        from google.oauth2.service_account import Credentials
        import gspread

        private_key = os.environ.get("GOOGLE_PRIVATE_KEY", "")
        if not private_key:
            print("GOOGLE_PRIVATE_KEY env var not set — Sheets logging disabled.")
            return None

        service_account_info = {
            "type": "service_account",
            "project_id": "animated-rhythm-449415-u3",
            "private_key_id": os.environ.get("GOOGLE_PRIVATE_KEY_ID", ""),
            "private_key": private_key.replace("\\n", "\n"),
            "client_email": "alphaearth-sheets-writer@animated-rhythm-449415-u3.iam.gserviceaccount.com",
            "client_id": os.environ.get("GOOGLE_CLIENT_ID", ""),
            "auth_uri": "https://accounts.google.com/o/oauth2/auth",
            "token_uri": "https://oauth2.googleapis.com/token",
            "auth_provider_x509_cert_url": "https://www.googleapis.com/oauth2/v1/certs",
            "client_x509_cert_url": "https://www.googleapis.com/robot/v1/metadata/x509/alphaearth-sheets-writer%40animated-rhythm-449415-u3.iam.gserviceaccount.com"
        }

        scopes = [
            'https://www.googleapis.com/auth/spreadsheets',
            'https://www.googleapis.com/auth/drive'
        ]

        credentials = Credentials.from_service_account_info(service_account_info, scopes=scopes)
        gc = gspread.authorize(credentials)
        return gc

    except Exception as e:
        print(f"Google Sheets setup failed: {str(e)}")
        return None

def test_sheets_connection():
    """Test if Google Sheets connection works"""
    try:
        gc = setup_google_sheets()
        if gc is None:
            print("❌ Sheets client not configured")
            return False

        sheet = gc.open("AlphaEarth User Interactions").sheet1
        print("✅ Successfully connected to Google Sheets")

        sheet.append_row(["test", "connection", "working"])
        print("✅ Test data written successfully")
        return True

    except Exception as e:
        print(f"❌ Connection test failed: {str(e)}")
        return False

# ===============================================
# APP CLASS DEFINITION WITH GOOGLE SHEETS LOGGING
# ===============================================

class AlphaEarthApp:
    def __init__(self, project_id):
        self.project_id = project_id

        self.analysis_results = None
        self.selected_bbox = None
        self.last_row_number = None
        self._ee_cache = {}
        self.run_history = []

        self.setup_constants()
        self.create_widgets()

        self.sheets_client = setup_google_sheets()

    def setup_constants(self):
        """Set up all constants and mappings"""
        self.ESA_CLASSES = {
            10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
            50: "Built-up", 60: "Bare/sparse", 70: "Snow/ice", 80: "Water",
            90: "Herb. wetland", 95: "Mangroves", 100: "Moss/lichen",
            999: "All other classes"
        }

        self.CLASS_COLORS = {
            10: '#006400', 20: '#ffbb22', 30: '#ffff4c', 40: '#f096ff',
            50: '#fa0000', 60: '#b4b4b4', 70: '#f0f0f0', 80: '#0064c8',
            90: '#0096a0', 95: '#00cf75', 100: '#fae6a0',
            999: '#808080'
        }

        self.COUNTRIES = [
            "Select your current country", "Afghanistan", "Albania", "Algeria", "Andorra",
            "Angola", "Antigua and Barbuda", "Argentina", "Armenia", "Australia",
            "Austria", "Azerbaijan", "Bahamas", "Bahrain", "Bangladesh",
            "Barbados", "Belarus", "Belgium", "Belize", "Benin",
            "Bhutan", "Bolivia", "Bosnia and Herzegovina", "Botswana", "Brazil",
            "Brunei", "Bulgaria", "Burkina Faso", "Burundi", "Cabo Verde",
            "Cambodia", "Cameroon", "Canada", "Central African Republic", "Chad",
            "Chile", "China", "Colombia", "Comoros", "Congo (Democratic Republic of)",
            "Congo (Republic of)", "Costa Rica", "Croatia", "Cuba", "Cyprus",
            "Czech Republic", "Denmark", "Djibouti", "Dominica", "Dominican Republic",
            "Ecuador", "Egypt", "El Salvador", "Equatorial Guinea", "Eritrea",
            "Estonia", "Eswatini", "Ethiopia", "Fiji", "Finland",
            "France", "Gabon", "Gambia", "Georgia", "Germany",
            "Ghana", "Greece", "Grenada", "Guatemala", "Guinea",
            "Guinea-Bissau", "Guyana", "Haiti", "Honduras", "Hungary",
            "Iceland", "India", "Indonesia", "Iran", "Iraq",
            "Ireland", "Israel", "Italy", "Jamaica", "Japan",
            "Jordan", "Kazakhstan", "Kenya", "Kiribati", "Kuwait",
            "Kyrgyzstan", "Laos", "Latvia", "Lebanon", "Lesotho",
            "Liberia", "Libya", "Liechtenstein", "Lithuania", "Luxembourg",
            "Madagascar", "Malawi", "Malaysia", "Maldives", "Mali",
            "Malta", "Marshall Islands", "Mauritania", "Mauritius", "Mexico",
            "Micronesia", "Moldova", "Monaco", "Mongolia", "Montenegro",
            "Morocco", "Mozambique", "Myanmar", "Namibia", "Nauru",
            "Nepal", "Netherlands", "New Zealand", "Nicaragua", "Niger",
            "Nigeria", "North Korea", "North Macedonia", "Norway", "Oman",
            "Pakistan", "Palau", "Palestine", "Panama", "Papua New Guinea",
            "Paraguay", "Peru", "Philippines", "Poland", "Portugal",
            "Qatar", "Romania", "Russia", "Rwanda", "Saint Kitts and Nevis",
            "Saint Lucia", "Saint Vincent and the Grenadines", "Samoa", "San Marino", "Sao Tome and Principe",
            "Saudi Arabia", "Senegal", "Serbia", "Seychelles", "Sierra Leone",
            "Singapore", "Slovakia", "Slovenia", "Solomon Islands", "Somalia",
            "South Africa", "South Korea", "South Sudan", "Spain", "Sri Lanka",
            "Sudan", "Suriname", "Sweden", "Switzerland", "Syria",
            "Tajikistan", "Tanzania", "Thailand", "Timor-Leste", "Togo",
            "Tonga", "Trinidad and Tobago", "Tunisia", "Turkey", "Turkmenistan",
            "Tuvalu", "Uganda", "Ukraine", "United Arab Emirates", "United Kingdom",
            "United States", "Uruguay", "Uzbekistan", "Vanuatu", "Vatican City",
            "Venezuela", "Vietnam", "Yemen", "Zambia", "Zimbabwe",
        ]

    def log_interaction_to_sheets(self, results_data):
        """Log user interaction data to Google Sheets with feedback as last column"""
        from datetime import datetime

        now = datetime.now()
        formatted_timestamp = f"{now.year} {now.month:02d} {now.day:02d} {now.hour:02d}"

        current_country = self.country_dropdown.value if self.country_dropdown.value != "Select your current country" else "Not selected"

        bbox = self.selected_bbox
        width = abs(bbox[2] - bbox[0])
        height = abs(bbox[3] - bbox[1])
        area = width * height

        _cb = results_data.get('class_b',
              list(self.class_b.value)[0] if self.class_b.value else 0)
        base_row_data = [
            formatted_timestamp,
            now.isoformat(),
            current_country,
            bbox[0], bbox[1], bbox[2], bbox[3],
            results_data['center_lat'],
            results_data['center_lon'],
            width, height, area,
            self.class_a.value,
            self.ESA_CLASSES[self.class_a.value],
            _cb,
            self.ESA_CLASSES.get(_cb, str(_cb)),
            self.algorithm.value,
            self.test_size.value,
            self.n_samples.value,
            self.scale_m.value,
            self.seed.value,
            results_data['accuracy'],
            results_data['roc_auc'],
            results_data['precision_a'],
            results_data['recall_a'],
            results_data['f1_a'],
            results_data['precision_b'],
            results_data['recall_b'],
            results_data['f1_b'],
            results_data['n_train'],
            results_data['n_test'],
            results_data['top_embeddings'][0][0] if len(results_data['top_embeddings']) > 0 else '',
            results_data['top_embeddings'][0][1] if len(results_data['top_embeddings']) > 0 else 0,
        ]

        for i in range(1, 5):
            if i < len(results_data['top_embeddings']):
                base_row_data.extend([
                    results_data['top_embeddings'][i][0],
                    results_data['top_embeddings'][i][1]
                ])
            else:
                base_row_data.extend(['', 0])

        all_embedding_data = []
        if 'all_embeddings' in results_data:
            for embedding_name, importance in results_data['all_embeddings']:
                all_embedding_data.append(embedding_name)
                all_embedding_data.append(importance)
        else:
            for i in range(1, 65):
                all_embedding_data.append(f'A{i:02d}')
                all_embedding_data.append(0.0)

        row_data = base_row_data + all_embedding_data + [""]

        try:
            if self.sheets_client is None:
                return

            sheet = self.sheets_client.open(GOOGLE_SHEET_NAME).sheet1

            try:
                existing_data = sheet.get_all_values()
                if len(existing_data) == 0:
                    base_headers = [
                        'timestamp', 'timestamp_iso', 'current_country',
                        'min_lon', 'min_lat', 'max_lon', 'max_lat',
                        'center_lat', 'center_lon', 'roi_width_degrees', 'roi_height_degrees', 'roi_area_square_degrees',
                        'class_a', 'class_a_name', 'class_b', 'class_b_name',
                        'algorithm', 'test_size_percent', 'samples_per_class', 'scale_meters', 'seed',
                        'accuracy', 'roc_auc', 'precision_class_a', 'recall_class_a', 'f1_class_a',
                        'precision_class_b', 'recall_class_b', 'f1_class_b', 'n_train', 'n_test',
                        'top_embedding_1', 'top_embedding_1_pct', 'top_embedding_2', 'top_embedding_2_pct',
                        'top_embedding_3', 'top_embedding_3_pct', 'top_embedding_4', 'top_embedding_4_pct',
                        'top_embedding_5', 'top_embedding_5_pct'
                    ]

                    embedding_headers = []
                    for i in range(1, 65):
                        embedding_headers.append(f'A{i:02d}_name')
                        embedding_headers.append(f'A{i:02d}_importance')

                    headers = base_headers + embedding_headers + ['user_feedback']
                    sheet.append_row(headers)
            except:
                pass

            sheet.append_row(row_data)

            current_rows = len(sheet.get_all_values())
            self.last_row_number = current_rows

        except Exception as e:
            print(f"Failed to log to Google Sheets: {str(e)}")
            print("Data logging disabled for this session")

    def update_feedback_in_sheets(self, feedback_text):
        """Update the most recent row with feedback data in the LAST column"""
        try:
            if self.sheets_client is None or self.last_row_number is None:
                return

            sheet = self.sheets_client.open(GOOGLE_SHEET_NAME).sheet1
            feedback_column = 170
            sheet.update_cell(self.last_row_number, feedback_column, feedback_text)

        except Exception as e:
            print(f"Failed to update feedback in Google Sheets: {str(e)}")

    def ee_pairwise_sample_with_others(self, class_a, class_b, *, year, region, scale_m, n_per_class, seed):
        """Sample data from Earth Engine with caching by bbox+class+scale+seed+year"""
        bbox = self.selected_bbox
        cache_key = (tuple(bbox) if bbox else None, class_a, class_b, scale_m, n_per_class, seed, year)
        if cache_key in self._ee_cache:
            if not getattr(self, "_cache_msg_printed", False):
                print("Using cached EE data...")
                self._cache_msg_printed = True
            return self._ee_cache[cache_key]

        emb, bands, emb_mask, esa = load_stack(year)

        all_esa_classes = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]

        if class_a == 999 and class_b == 999:
            raise ValueError("Both classes cannot be 'All other classes'")

        elif class_a == 999:
            specific_class = class_b
            other_classes = [c for c in all_esa_classes if c != specific_class]
            others_mask = esa.eq(other_classes[0])
            for other_class in other_classes[1:]:
                others_mask = others_mask.Or(esa.eq(other_class))
            specific_mask = esa.eq(specific_class)
            pair_mask = others_mask.Or(specific_mask)
            relabeled_esa = esa.where(others_mask, 999)

        elif class_b == 999:
            specific_class = class_a
            other_classes = [c for c in all_esa_classes if c != specific_class]
            others_mask = esa.eq(other_classes[0])
            for other_class in other_classes[1:]:
                others_mask = others_mask.Or(esa.eq(other_class))
            specific_mask = esa.eq(specific_class)
            pair_mask = others_mask.Or(specific_mask)
            relabeled_esa = esa.where(others_mask, 999)

        else:
            pair_mask = esa.eq(class_a).Or(esa.eq(class_b))
            relabeled_esa = esa

        stack = (emb.updateMask(emb_mask).updateMask(pair_mask)
                ).addBands(relabeled_esa.updateMask(emb_mask).updateMask(pair_mask).rename('label'))

        pair_fc = stack.stratifiedSample(
            numPoints=n_per_class * 2,
            classBand='label',
            region=region,
            scale=scale_m,
            classValues=[class_a, class_b],
            classPoints=[n_per_class, n_per_class],
            seed=seed,
            geometries=True
        ).select(bands.cat(ee.List(['label'])))

        size = pair_fc.size().getInfo()
        hist = pair_fc.aggregate_histogram('label').getInfo()

        props = bands.cat(ee.List(['label']))
        attr_fc = pair_fc.map(lambda f: ee.Feature(None, f.toDictionary(props)))
        feats = attr_fc.getInfo()['features'] if size else []
        df = pd.DataFrame([f['properties'] for f in feats]) if feats else pd.DataFrame(columns=[*bands.getInfo(), 'label'])

        result = (df, list(bands.getInfo()), hist, size, pair_fc)
        self._ee_cache[cache_key] = result
        return result

    def create_widgets(self):
        """Create all UI widgets with interactive map for region selection"""
        self.title = widgets.HTML(
            value="""
            <div style='text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        color: white; border-radius: 10px; margin-bottom: 20px;'>
                <h1 style='margin: 0; font-size: 28px;'>Google AlphaEarth Foundations App for Land Cover Classification</h1>

            </div>
            """
        )

        self.welcome_box = widgets.HTML(
            value=f"""
            <div style='background: #f8f9ff; padding: 20px; border-radius: 10px; margin-bottom: 20px;
                        border-left: 4px solid #4f46e5; color: #1e293b;'>
                <p style='margin: 0 0 15px 0; font-size: 16px; line-height: 1.6;'>
                    <span style='color: #dc2626; font-weight: bold;'>Welcome, please read before running the App:</span>
                    this app uses the <strong><a href="https://deepmind.google/discover/blog/alphaearth-foundations-helps-map-our-planet-in-unprecedented-detail/" target="_blank" style='color: #2563eb; text-decoration: none;'>Google AlphaEarth AI-Foundation Model</a></strong> for Land Cover Classification tasks.
                    You will interact face to face with the 64 embeddings of this model to classify
                    land cover types using validated label data from the European Space Agency (2020).
                </p>
                <p style='margin: 0 0 10px 0; font-size: 16px; font-weight: 500; color: #1e40af;'>
                    For example, you can compare:
                </p>
                <ul style='margin: 0 0 15px 20px; font-size: 15px; line-height: 1.5; color: #2563eb;'>
                    <li style='color: #2563eb; font-weight: 500;'><strong>Built-up vs Water</strong></li>
                    <li style='color: #2563eb; font-weight: 500;'><strong>Cropland vs Grassland</strong></li>
                    <li style='color: #2563eb; font-weight: 500;'><strong>Shrubland vs Tree cover</strong></li>
                    <li style='color: #2563eb; font-weight: 500;'><strong>or select among 11 land cover types (55 possible combinations)</strong></li>
                </ul>
                <p style='margin: 0 0 15px 0; font-size: 16px; line-height: 1.6; color: #1e293b;'>
                    <strong style='color: #7c2d12;'>🌍 Research Data Collection:</strong> Each time you run a classification,
                    your interaction data will be automatically stored for research purposes. No personal information is registered.
                </p>
                <div style='background: #e0f2fe; padding: 15px; border-radius: 8px; border-left: 4px solid #0288d1;'>
                    <h4 style='margin: 0 0 10px 0; font-size: 16px; font-weight: bold; color: #01579b;'>Instructions:</h4>
                    <ol style='margin: 0; font-size: 15px; line-height: 1.6; color: #1e293b;'>
                        <li style='margin-bottom: 6px;'><strong>Select your current country</strong> (map auto-centers to that country)</li>
                        <li style='margin-bottom: 6px;'><strong>Select a Region of Interest (ROI)</strong> by drawing on the map, entering coordinates, or using the location search box</li>
                        <li style='margin-bottom: 6px;'><strong>Select land cover classes</strong> for classification</li>
                        <li style='margin-bottom: 6px;'><strong>Select Algorithm Settings</strong> (single or All Algorithms, year range 2017–2024, temporal resolution)</li>
                        <li style='margin-bottom: 6px;'><strong>Hit the RUN ANALYSIS button</strong></li>
                        <li style='margin-bottom: 6px;'><strong>Check results</strong>, comparison table, and chart</li>
                        <li style='margin-bottom: 6px;'><strong>Download</strong> embeddings, CSV, comparison table, or time series data</li>
                        <li style='margin-bottom: 6px;'><strong>Provide feedback</strong></li>
                    </ol>
                </div>
            </div>
            """
        )

        self.attribution_box = widgets.HTML(
            value="""
            <div style='background: #f1f5f9; padding: 20px; border-radius: 8px; margin-bottom: 20px;
                        text-align: center; border-left: 4px solid #64748b;'>
                <p style='margin: 0; font-size: 14px; color: #475569; line-height: 1.5;'>
                    This App was created by <strong>Felipe Benavides</strong> from the SDS Lab at<br>
                    <strong>Northeastern University</strong> and the <strong>Gulf of Maine Research Institute</strong><br>
                    <a href="mailto:pipeben@gmail.com" style="color: #0ea5e9; text-decoration: none; font-weight: 500;">pipeben@gmail.com</a>
                </p>
                <div style='border-top: 1px solid #cbd5e1; margin: 15px 0 5px 0; padding-top: 15px;'>
                    <p style='margin: 0; font-size: 12px; color: #64748b; line-height: 1.4; font-style: italic;'>
                        Benavides, F. (2025). AlphaEarth Foundation Model for Land Cover Classification App.
                        SDS Lab, Northeastern University and Gulf of Maine Research Institute.
                        <em>Zenodo</em>. <a href="https://doi.org/10.5281/zenodo.16911104" target="_blank" style="color: #2563eb; text-decoration: none;">https://doi.org/10.5281/zenodo.16911104</a>
                    </p>
                </div>
                <div style='border-top: 1px solid #cbd5e1; margin: 10px 0 0 0; padding-top: 12px;'>
                    <p style='margin: 0; font-size: 13px; color: #475569; line-height: 1.5;'>
                        Extended by <strong>Mahat Maalim Ibrahim</strong>, School of Business, Department of Economics,<br>
                        <strong>Ibn Haldun University</strong>, Istanbul, Turkey.<br>
                        <a href="mailto:mahatibrahim@ihu.edu.tr" style="color: #0ea5e9; text-decoration: none; font-weight: 500;">mahatibrahim@ihu.edu.tr</a>
                        &nbsp;|&nbsp;
                        <a href="mailto:mahatmaalimibrahim@gmail.com" style="color: #0ea5e9; text-decoration: none; font-weight: 500;">mahatmaalimibrahim@gmail.com</a>
                    </p>
                </div>
            </div>
            """
        )

        self.country_title = widgets.HTML(
            "<h3 style='color: #2c3e50; margin: 20px 0 10px 0;'>User Information</h3>"
        )

        self.country_dropdown = widgets.Dropdown(
            options=self.COUNTRIES,
            value="Select your current country",
            description='What country are you now?',
            style={'description_width': '170px'},
            layout=widgets.Layout(width='500px')
        )

        # Region selection section
        self.region_title = widgets.HTML(
            "<h3 style='color: #2c3e50; margin: 20px 0 10px 0;'>Region of Interest (ROI) selection</h3>"
        )

        self.region_help = widgets.HTML(
            value="""
            <div style='background: #d1ecf1; padding: 15px; border-radius: 8px; margin: 10px 0;
                        border-left: 4px solid #0ea5e9; color: #2c3e50; font-size: 14px;'>
                <b style='color: #0c4a6e;'>How to select your region:</b><br>
                <span style='color: #374151;'>1. Use the <b>Search box</b> to jump to a location</span><br>
                <span style='color: #374151;'>2. Click the rectangle or polygon tool to draw your ROI</span><br>
                <span style='color: #374151;'>3. Or enter coordinates manually below the map and click Apply</span><br>
                <span style='color: #374151;'>4. Use the trash tool to delete shapes</span>
            </div>
            """
        )

        # Location search above map
        self.search_input = widgets.Text(
            placeholder='Search location (e.g. "Paris" or "Amazon rainforest")',
            description='Location:',
            style={'description_width': '70px'},
            layout=widgets.Layout(width='400px')
        )
        self.go_button = widgets.Button(
            description='Go',
            button_style='info',
            layout=widgets.Layout(width='60px', height='32px')
        )
        self.search_status = widgets.HTML()
        self.go_button.on_click(self.go_to_location)

        # Interactive map with drawing tools
        self.selection_map = Map(
            basemap=basemaps.Esri.WorldImagery,
            center=[41.8, -72.6],
            zoom=7,
            scroll_wheel_zoom=True,
            layout=widgets.Layout(height='350px', width='100%')
        )

        # Drawing control with both rectangle and polygon enabled
        self.draw_control = DrawControl(
            marker={},
            circle={},
            circlemarker={},
            polyline={},
            polygon={
                "shapeOptions": {
                    "fillColor": "blue",
                    "color": "blue",
                    "fillOpacity": 0.2,
                    "weight": 3
                }
            },
            rectangle={
                "shapeOptions": {
                    "fillColor": "red",
                    "color": "red",
                    "fillOpacity": 0.3,
                    "weight": 3
                }
            }
        )

        self.draw_control.on_draw(self.handle_draw)
        self.selection_map.add_control(self.draw_control)

        # Coordinate input boxes below the map
        self.coord_title = widgets.HTML(
            "<b style='color:#2c3e50; font-size:14px;'>Or enter coordinates manually:</b>"
        )
        self.min_lon_input = widgets.FloatText(
            value=-73.0, description='Min Lon:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='160px')
        )
        self.min_lat_input = widgets.FloatText(
            value=41.5, description='Min Lat:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='160px')
        )
        self.max_lon_input = widgets.FloatText(
            value=-72.0, description='Max Lon:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='160px')
        )
        self.max_lat_input = widgets.FloatText(
            value=42.2, description='Max Lat:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='160px')
        )
        self.apply_bbox_btn = widgets.Button(
            description='Apply',
            button_style='warning',
            layout=widgets.Layout(width='80px', height='32px')
        )
        self.apply_bbox_btn.on_click(self.apply_bbox_from_inputs)

        # File upload widget for shapefile/GeoJSON
        self.upload_widget = widgets.FileUpload(
            accept='.zip,.geojson',
            multiple=False,
            description='Upload Study Area',
            layout=widgets.Layout(width='220px')
        )
        self.upload_status = widgets.HTML()
        self.upload_layer = None
        self.upload_geojson = None
        self.upload_widget.observe(self.handle_upload, names='value')

        # Region info display
        self.region_info = widgets.HTML()
        self.region_status = widgets.HTML(
            value="""
            <div style='background: #fff3cd; padding: 10px; border-radius: 8px; margin: 10px 0;
                        border-left: 4px solid #f59e0b; color: #2c3e50; font-size: 14px;'>
                <b style='color: #92400e;'>No region selected</b><br>
                <span style='color: #374151;'>Please draw a shape on the map or enter coordinates below.</span>
            </div>
            """
        )

        # Class selection
        self.class_title = widgets.HTML(
            "<h3 style='color: #2c3e50; margin: 20px 0 10px 0;'>Land Cover Classes</h3>"
        )

        class_options = [(f"{k} - {v}", k) for k, v in self.ESA_CLASSES.items()]

        self.class_a = widgets.Dropdown(
            options=class_options, value=10, description='Class A:',
            style={'description_width': 'initial'}
        )

        self.class_b = widgets.SelectMultiple(
            options=class_options, value=(80,), description='Class B:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(height='180px', width='350px')
        )
        self.class_b_label = widgets.HTML(
            "<span style='font-size:12px;color:#64748b;font-style:italic;'>"
            "Hold Ctrl (Windows/Linux) or Cmd (Mac) to select multiple classes</span>"
        )

        self.class_comparison = widgets.HTML()

        # Algorithm settings
        self.algo_title = widgets.HTML(
            "<h3 style='color: #2c3e50; margin: 20px 0 10px 0;'>Algorithm Settings</h3>"
        )

        self.algorithm = widgets.Dropdown(
            options=[('Random Forest', 'rf'), ('Gradient Boosting', 'gbt'),
                    ('XGBoost', 'xgb'), ('LightGBM', 'lgb'),
                    ('All Algorithms', 'all')],
            value='rf', description='Algorithm:', style={'description_width': 'initial'}
        )

        self.test_size = widgets.IntSlider(
            value=25, min=10, max=40, step=5, description='Test Data %:',
            style={'description_width': 'initial'}
        )

        self.n_samples = widgets.IntSlider(
            value=100, min=50, max=5000, step=100, description='Samples/class:',
            style={'description_width': 'initial'}
        )

        self.scale_m = widgets.IntSlider(
            value=500, min=100, max=2000, step=100, description='Scale (m):',
            style={'description_width': 'initial'}
        )

        self.seed = widgets.IntText(value=42, description='Seed:', style={'description_width': 'initial'})

        # Year From / Year To dropdowns (2017-2024)
        self.year_from_dropdown = widgets.Dropdown(
            options=list(range(2017, 2025)),
            value=2020,
            description='Year From:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='210px')
        )
        self.year_to_dropdown = widgets.Dropdown(
            options=list(range(2017, 2025)),
            value=2023,
            description='Year To:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='210px')
        )

        # Download N dropdown (3, 5, or 10 embeddings)
        self.download_n_dropdown = widgets.Dropdown(
            options=[('Top 3', 3), ('Top 5', 5), ('Top 10', 10)],
            value=5,
            description='Show top N:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )

        # Temporal resolution for time series download
        self.temporal_res_dropdown = widgets.Dropdown(
            options=['Yearly', 'Monthly', 'Daily'],
            value='Yearly',
            description='Temporal Res:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )
        _month_opts = [(m, i + 1) for i, m in enumerate([
            'January', 'February', 'March', 'April', 'May', 'June',
            'July', 'August', 'September', 'October', 'November', 'December'
        ])]
        self.month_from_dropdown = widgets.Dropdown(
            options=_month_opts, value=1, description='Month From:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px', display='none')
        )
        self.month_to_dropdown = widgets.Dropdown(
            options=_month_opts, value=12, description='Month To:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px', display='none')
        )
        self.day_from_picker = widgets.Text(
            placeholder='YYYY-MM-DD', description='Day From:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px', display='none')
        )
        self.day_to_picker = widgets.Text(
            placeholder='YYYY-MM-DD', description='Day To:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px', display='none')
        )
        def _on_temporal_res(change):
            v = change['new']
            self.month_from_dropdown.layout.display = 'flex' if v == 'Monthly' else 'none'
            self.month_to_dropdown.layout.display = 'flex' if v == 'Monthly' else 'none'
            self.day_from_picker.layout.display = 'flex' if v == 'Daily' else 'none'
            self.day_to_picker.layout.display = 'flex' if v == 'Daily' else 'none'
        self.temporal_res_dropdown.observe(_on_temporal_res, names='value')
        # Run button
        self.run_button = widgets.Button(
            description='Select country and draw region first',
            button_style='',
            disabled=True,
            layout=widgets.Layout(width='350px', height='50px'),
        )

        # Progress bar
        self.progress = widgets.IntProgress(
            value=0, min=0, max=100,
            description='Progress:',
            bar_style='info',
            style={'bar_color': '#3b82f6', 'description_width': 'initial'},
            layout=widgets.Layout(width='400px', margin='10px 0', visibility='hidden')
        )

        self.status = widgets.HTML()
        self.output_area = widgets.Output()

        # Feedback section
        self.feedback_title = widgets.HTML(
            "<h3 style='color: #2c3e50; margin: 30px 0 10px 0;'>Feedback</h3>"
        )

        self.feedback_question = widgets.HTML(
            value="""
            <div style='background: #f0fdf4; padding: 15px; border-radius: 8px; margin: 10px 0;
                        border-left: 4px solid #16a34a; color: #1e293b;'>
                <h4 style='margin: 0 0 10px 0; color: #166534; font-size: 16px;'>
                    How has this app helped you to get familiar with Google AlphaEarth embeddings?
                </h4>
            </div>
            """
        )

        self.feedback_textarea = widgets.Textarea(
            value='',
            placeholder='Share your experience with the AlphaEarth embeddings...',
            description='Your feedback:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='100%', height='120px')
        )

        self.feedback_button = widgets.Button(
            description='Submit Feedback',
            button_style='success',
            layout=widgets.Layout(width='200px', height='40px'),
            disabled=False
        )

        self.feedback_status = widgets.HTML()

        self.feedback_section = widgets.VBox([
            self.feedback_title,
            self.feedback_question,
            self.feedback_textarea,
            widgets.HBox([self.feedback_button], layout=widgets.Layout(justify_content='center')),
            self.feedback_status
        ], layout=widgets.Layout(visibility='hidden', margin='20px 0'))

        # Results history output
        self.history_output = widgets.Output()

        # Bind events
        self.run_button.on_click(self.run_analysis)
        self.feedback_button.on_click(self.submit_feedback)

        for widget in [self.class_a, self.class_b]:
            widget.observe(self.update_class_comparison, names='value')

        self.country_dropdown.observe(self.update_button_state, names='value')
        self.country_dropdown.observe(self.on_country_changed, names='value')

        self.update_class_comparison(None)
        self.update_button_state(None)

    def go_to_location(self, b):
        """Geocode the search input and recenter the map"""
        location_name = self.search_input.value.strip()
        if not location_name:
            self.search_status.value = "<span style='color:red;'>Enter a location name first.</span>"
            return
        try:
            geolocator = _get_nominatim()(user_agent="alphaearth_app")
            location = geolocator.geocode(location_name, timeout=10)
            if location is None:
                self.search_status.value = f"<span style='color:red;'>Location not found: {location_name}</span>"
                return
            self.selection_map.center = [location.latitude, location.longitude]
            self.selection_map.zoom = 8
            self.search_status.value = f"<span style='color:green;'>Moved to: {location.address[:60]}...</span>"
        except Exception as e:
            self.search_status.value = f"<span style='color:red;'>Search failed: {str(e)}</span>"

    def on_country_changed(self, change):
        """Auto-center the selection map when a country is selected."""
        country = change['new']
        if country == "Select your current country":
            return
        try:
            geolocator = _get_nominatim()(user_agent="alphaearth_app")
            location = geolocator.geocode(country, timeout=10)
            if location:
                self.selection_map.center = [location.latitude, location.longitude]
                self.selection_map.zoom = 5
        except Exception:
            pass

    def apply_bbox_from_inputs(self, b):
        """Apply manually entered coordinates as selected_bbox"""
        try:
            bbox = [
                float(self.min_lon_input.value),
                float(self.min_lat_input.value),
                float(self.max_lon_input.value),
                float(self.max_lat_input.value)
            ]
            if bbox[0] >= bbox[2]:
                self.region_status.value = "<div style='color:red;padding:8px;'>Min Lon must be less than Max Lon.</div>"
                return
            if bbox[1] >= bbox[3]:
                self.region_status.value = "<div style='color:red;padding:8px;'>Min Lat must be less than Max Lat.</div>"
                return
            self.selected_bbox = bbox
            if b is not None:
                self.upload_geojson = None
            self.update_region_info()
            self.update_button_state(None)
            print(f"Coordinates applied: {[round(b, 3) for b in bbox]}")
        except Exception as e:
            self.region_status.value = f"<div style='color:red;padding:8px;'>Invalid coordinates: {str(e)}</div>"

    def handle_upload(self, change):
        """Handle shapefile (.zip) or GeoJSON upload to set bounding box"""
        if not change['new']:
            return
        try:
            try:
                import geopandas as gpd
            except ImportError:
                import subprocess, sys
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'geopandas', '--quiet'])
                import geopandas as gpd
            import io, zipfile, tempfile, os
            uploaded = change['new']
            # Support ipywidgets 7.x (dict) and 8.x (tuple of dicts)
            if isinstance(uploaded, dict):
                fname = next(iter(uploaded))
                content = bytes(uploaded[fname]['content'])
            else:
                item = uploaded[0]
                fname = item['name']
                content = bytes(item['content'])
            if fname.lower().endswith('.geojson'):
                gdf = gpd.read_file(io.BytesIO(content))
            elif fname.lower().endswith('.zip'):
                with tempfile.TemporaryDirectory() as tmpdir:
                    zip_path = os.path.join(tmpdir, fname)
                    with open(zip_path, 'wb') as zf:
                        zf.write(content)
                    with zipfile.ZipFile(zip_path, 'r') as z:
                        z.extractall(tmpdir)
                    shp_files = [os.path.join(tmpdir, fn) for fn in os.listdir(tmpdir) if fn.endswith('.shp')]
                    if not shp_files:
                        self.upload_status.value = "<div style='color:red;padding:8px;'>No .shp file found inside the zip.</div>"
                        return
                    gdf = gpd.read_file(shp_files[0])
            else:
                self.upload_status.value = "<div style='color:red;padding:8px;'>Unsupported file type. Use .zip or .geojson.</div>"
                return
            minx, miny, maxx, maxy = gdf.total_bounds
            self.min_lon_input.value = round(float(minx), 6)
            self.min_lat_input.value = round(float(miny), 6)
            self.max_lon_input.value = round(float(maxx), 6)
            self.max_lat_input.value = round(float(maxy), 6)
            self.apply_bbox_from_inputs(None)
            # Remove previous upload layer then draw new boundary
            if self.upload_layer is not None:
                try:
                    self.selection_map.remove_layer(self.upload_layer)
                except Exception:
                    pass
            geo_data = gdf.to_crs(epsg=4326).__geo_interface__
            self.upload_geojson = geo_data
            self.upload_layer = GeoJSON(
                data=geo_data,
                style={'color': '#2563eb', 'weight': 2,
                       'fillOpacity': 0.05, 'fillColor': '#2563eb'}
            )
            self.selection_map.add_layer(self.upload_layer)
            self.selection_map.fit_bounds(
                [[float(miny), float(minx)], [float(maxy), float(maxx)]]
            )
            bb_str = '[' + ', '.join([str(round(float(v), 4)) for v in [minx, miny, maxx, maxy]]) + ']'
            self.upload_status.value = (
                "<div style='background:#dcfce7;border-left:4px solid #16a34a;"
                "padding:10px 14px;border-radius:6px;margin:6px 0;"
                "font-size:13px;color:#14532d;'>"
                + '\u2705 <b>Loaded: ' + fname + '</b>'
                + ' - Bounding box applied: ' + bb_str
                + "</div>"
            )
        except Exception as e:
            self.upload_status.value = "<div style='color:red;padding:8px;'>Upload error: " + str(e) + "</div>"

    def update_button_state(self, change):
        """Update run button state based on country selection and region drawing"""
        country_selected = self.country_dropdown.value != "Select your current country"
        region_selected = self.selected_bbox is not None

        if country_selected and region_selected:
            self.run_button.disabled = False
            self.run_button.description = 'RUN ANALYSIS'
            self.run_button.button_style = 'primary'
        elif not country_selected:
            self.run_button.disabled = True
            self.run_button.description = 'Select country first'
            self.run_button.button_style = ''
        else:
            self.run_button.disabled = True
            self.run_button.description = 'Draw a region on map'
            self.run_button.button_style = ''

    def handle_draw(self, target, action, geo_json):
        """Handle rectangle or polygon drawing on the map"""
        if action == 'created' and geo_json['geometry']['type'] == 'Polygon':
            coords = geo_json['geometry']['coordinates'][0]
            lons = [coord[0] for coord in coords]
            lats = [coord[1] for coord in coords]

            bbox = [min(lons), min(lats), max(lons), max(lats)]
            self.selected_bbox = bbox
            self.upload_geojson = None
            self.min_lon_input.value = round(bbox[0], 6)
            self.min_lat_input.value = round(bbox[1], 6)
            self.max_lon_input.value = round(bbox[2], 6)
            self.max_lat_input.value = round(bbox[3], 6)

            self.update_region_info()
            self.update_button_state(None)

            print(f"Region selected! Bounds: {[round(b, 3) for b in bbox]}")

        elif action == 'deleted':
            self.selected_bbox = None
            self.region_info.value = ""
            self.region_status.value = """
            <div style='background: #fff3cd; padding: 10px; border-radius: 8px; margin: 10px 0;
                        border-left: 4px solid #f59e0b; color: #2c3e50; font-size: 14px;'>
                <b style='color: #92400e;'>No region selected</b><br>
                <span style='color: #374151;'>Please draw a shape on the map or enter coordinates below.</span>
            </div>
            """
            self.update_button_state(None)

    def update_region_info(self):
        """Update region information display"""
        if self.selected_bbox:
            bbox = self.selected_bbox
            width = abs(bbox[2] - bbox[0])
            height = abs(bbox[3] - bbox[1])
            area = width * height

            self.region_info.value = f"""
            <div style='background: #f0f9ff; padding: 12px; border-radius: 8px; margin: 10px 0;
                        border-left: 4px solid #3b82f6; color: #1e293b; font-size: 14px;'>
                <b style='color: #1e40af; font-size: 15px;'>Selected Region:</b><br>
                <span style='color: #374151; font-weight: 500;'>Longitude: {bbox[0]:.3f} to {bbox[2]:.3f} ({width:.3f}°)</span><br>
                <span style='color: #374151; font-weight: 500;'>Latitude: {bbox[1]:.3f} to {bbox[3]:.3f} ({height:.3f}°)</span><br>
                <span style='color: #374151; font-weight: 500;'>Area: {area:.4f} square degrees</span>
            </div>
            """

            self.region_status.value = ""

    def update_class_comparison(self, change):
        """Update class comparison display"""
        try:
            class_b_vals = list(self.class_b.value) if self.class_b.value else []
            if not class_b_vals:
                self.class_comparison.value = (
                    "<div style='background:#fff3cd;padding:10px;border-radius:8px;"
                    "border-left:4px solid #f59e0b;color:#92400e;font-size:14px;'>"
                    "<b>No Class B selected.</b> Please select at least one.</div>"
                )
                return
            class_a_val = self.class_a.value
            if class_a_val == 999 and all(v == 999 for v in class_b_vals):
                self.class_comparison.value = (
                    "<div style='background:#fee2e2;padding:12px;border-radius:8px;"
                    "border-left:4px solid #ef4444;color:#1e293b;font-size:14px;'>"
                    "<b style='color:#dc2626;'>Invalid:</b> Both classes cannot be "
                    "'All other classes'.</div>"
                )
                return
            class_a_name = self.ESA_CLASSES[class_a_val]
            if len(class_b_vals) == 1:
                class_b_val = class_b_vals[0]
                class_b_name = self.ESA_CLASSES[class_b_val]
                if class_a_val == 999 or class_b_val == 999:
                    difficulty = 0.5; diff_label = "Medium"; diff_color = "#d97706"
                else:
                    difficulty = self.calculate_class_similarity(class_a_val, class_b_val)
                    if difficulty > 0.6:
                        diff_label = "Hard"; diff_color = "#dc2626"
                    elif difficulty > 0.4:
                        diff_label = "Medium"; diff_color = "#d97706"
                    else:
                        diff_label = "Easy"; diff_color = "#059669"
                self.class_comparison.value = (
                    "<div style='background:#e8f5e8;padding:12px;border-radius:8px;"
                    "border-left:4px solid #10b981;color:#1e293b;font-size:14px;'>"
                    "<b style='color:#047857;'>Comparison:</b><br>"
                    "<span style='color:#374151;font-weight:500;'>"
                    + str(class_a_val) + " (" + class_a_name + ") vs "
                    + str(class_b_val) + " (" + class_b_name + ")</span><br>"
                    + "<span style='color:" + diff_color + ";font-weight:bold;'>"
                    + "Classification difficulty: " + diff_label + "</span></div>"
                )
            else:
                _pair_lines = []
                for _bv in class_b_vals:
                    if _bv == class_a_val:
                        continue
                    _bn = self.ESA_CLASSES.get(_bv, str(_bv))
                    if class_a_val == 999 or _bv == 999:
                        _dlabel = 'Medium'; _dcolor = '#d97706'
                    else:
                        _d = self.calculate_class_similarity(class_a_val, _bv)
                        if _d > 0.6:
                            _dlabel = 'Hard'; _dcolor = '#dc2626'
                        elif _d > 0.4:
                            _dlabel = 'Medium'; _dcolor = '#d97706'
                        else:
                            _dlabel = 'Easy'; _dcolor = '#059669'
                    _pair_lines.append(
                        "<span style='color:#374151;font-weight:500;'>" + class_a_name
                        + ' vs ' + _bn + "</span>"
                        + " &mdash; <span style='color:" + _dcolor + ";font-weight:bold;'>"
                        + "Classification difficulty: " + _dlabel + "</span>"
                    )
                self.class_comparison.value = (
                    "<div style='background:#e8f5e8;padding:12px;border-radius:8px;"
                    "border-left:4px solid #10b981;color:#1e293b;font-size:14px;'>"
                    "<b style='color:#047857;'>Multi-Class B Comparison:</b><br>"
                    + '<br>'.join(_pair_lines)
                    + "</div>"
                )
        except Exception:
            self.class_comparison.value = ""

    def calculate_class_similarity(self, class_a, class_b):
        """Calculate how similar two classes are"""
        similarity_scores = {
            (10, 20): 0.8, (20, 30): 0.7, (30, 40): 0.6, (90, 95): 0.9, (10, 95): 0.5,
            (20, 100): 0.4, (60, 100): 0.5, (50, 60): 0.4, (50, 80): 0.1, (70, 80): 0.15,
            (10, 50): 0.2, (10, 80): 0.2, (50, 70): 0.1, (40, 80): 0.25
        }
        key = tuple(sorted([class_a, class_b]))
        return similarity_scores.get(key, 0.35)

    def submit_feedback(self, button):
        """Handle feedback submission"""
        feedback_text = self.feedback_textarea.value.strip()

        if not feedback_text:
            self.feedback_status.value = """
            <div style='background: #fee2e2; padding: 10px; border-radius: 8px; margin: 10px 0; text-align: center;'>
                <span style='color: #dc2626; font-weight: bold;'>Please provide some feedback before submitting.</span>
            </div>
            """
            return

        self.update_feedback_in_sheets(feedback_text)

        self.feedback_status.value = """
        <div style='background: #dcfce7; padding: 12px; border-radius: 8px; margin: 10px 0; text-align: center;'>
            <span style='color: #166534; font-weight: bold; font-size: 16px;'>
                Thank you for your feedback! Your response has been recorded.
            </span>
        </div>
        """

        self.feedback_textarea.disabled = True
        self.feedback_button.disabled = True
        self.feedback_button.description = 'Feedback Submitted'

    def display_app(self):
        """Display the complete app interface"""
        custom_css = widgets.HTML("""
        <style>
            .widget-html-content { color: #1e293b !important; }
            .widget-text input, .widget-dropdown select {
                color: #1e293b !important;
                background-color: white !important;
            }
            .widget-label { color: #374151 !important; font-weight: 500 !important; }
        </style>
        """)

        app_layout = widgets.VBox([
            custom_css,
            self.title,
            self.welcome_box,
            self.attribution_box,

            # Country section
            self.country_title,
            self.country_dropdown,

            widgets.HTML("<hr style='border: 1px solid #e5e7eb; margin: 20px 0;'>"),

            # Upload Study Area section
            widgets.HTML("<h3 style='color: #2c3e50; margin: 20px 0 10px 0;'>Upload Study Area</h3>"),
            widgets.VBox([
                self.upload_widget,
                widgets.HTML("<span style='font-size:12px;color:#64748b;font-style:italic;'>"
                             "Supported formats: Zipped shapefile (.zip) or GeoJSON (.geojson)</span>"),
                self.upload_status,
            ], layout=widgets.Layout(margin='8px 0')),

            widgets.HTML("<hr style='border: 1px solid #e5e7eb; margin: 20px 0;'>"),

            # ROI section
            self.region_title,
            self.region_help,
            # Location search above map
            widgets.HBox([self.search_input, self.go_button, self.search_status],
                         layout=widgets.Layout(align_items='center', margin='5px 0')),
            self.selection_map,
            # Coordinate inputs below map
            widgets.VBox([
                self.coord_title,
                widgets.HBox([self.min_lon_input, self.min_lat_input,
                               self.max_lon_input, self.max_lat_input,
                               self.apply_bbox_btn],
                              layout=widgets.Layout(align_items='center', margin='5px 0'))
            ], layout=widgets.Layout(margin='8px 0')),
            self.region_status,
            self.region_info,

            widgets.HTML("<hr style='border: 1px solid #e5e7eb; margin: 20px 0;'>"),

            self.class_title,
            widgets.HBox([self.class_a, widgets.VBox([self.class_b, self.class_b_label])]),
            self.class_comparison,

            widgets.HTML("<hr style='border: 1px solid #e5e7eb; margin: 20px 0;'>"),

            self.algo_title,
            widgets.HBox([self.algorithm, self.test_size]),
            widgets.HBox([self.n_samples, self.scale_m]),
            widgets.HBox([self.seed, self.year_from_dropdown, self.year_to_dropdown, self.download_n_dropdown]),
            widgets.HBox([self.temporal_res_dropdown, self.month_from_dropdown,
                         self.month_to_dropdown, self.day_from_picker, self.day_to_picker],
                         layout=widgets.Layout(align_items='center', margin='5px 0')),

            widgets.HTML("<hr style='border: 1px solid #e5e7eb; margin: 20px 0;'>"),

            widgets.VBox([
                self.run_button,
                self.progress,
                self.status
            ], layout=widgets.Layout(align_items='center')),

            widgets.HTML("<hr style='border: 1px solid #e5e7eb; margin: 20px 0;'>"),

            self.output_area,

            # Feedback section
            self.feedback_section,

            widgets.HTML("<hr style='border: 1px solid #e5e7eb; margin: 20px 0;'>"),

            # Results history
            widgets.HTML("<h3 style='color: #2c3e50; margin: 10px 0;'>Session Results History</h3>"),
            self.history_output,
        ])

        display(app_layout)

    def ee_pairwise_sample_global(self, class_a, class_b, *, year, region, scale_m, n_per_class, seed):
        """Sample data from Earth Engine - from working code"""
        emb, bands, emb_mask, esa = load_stack(year)
        pair_mask = esa.eq(class_a).Or(esa.eq(class_b))
        stack = (emb.updateMask(emb_mask).updateMask(pair_mask)
                ).addBands(esa.updateMask(emb_mask).updateMask(pair_mask))

        pair_fc = stack.stratifiedSample(
            numPoints=n_per_class * 2,
            classBand='label',
            region=region,
            scale=scale_m,
            classValues=[class_a, class_b],
            classPoints=[n_per_class, n_per_class],
            seed=seed,
            geometries=True
        ).select(bands.cat(ee.List(['label'])))

        size = pair_fc.size().getInfo()
        hist = pair_fc.aggregate_histogram('label').getInfo()

        props = bands.cat(ee.List(['label']))
        attr_fc = pair_fc.map(lambda f: ee.Feature(None, f.toDictionary(props)))
        feats = attr_fc.getInfo()['features'] if size else []
        df = pd.DataFrame([f['properties'] for f in feats]) if feats else pd.DataFrame(columns=[*bands.getInfo(), 'label'])
        return df, list(bands.getInfo()), hist, size, pair_fc

    def build_model(self, name, random_state=42):
        """Build ML model - from working code"""
        name = name.lower()
        if name == "rf":
            return RandomForestClassifier(n_estimators=300, random_state=random_state, n_jobs=-1)
        elif name == "gbt":
            return GradientBoostingClassifier(random_state=random_state)
        elif name == "lgb":
            lgb = _get_lgb()
            return lgb.LGBMClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                objective='binary', metric='binary_logloss',
                n_jobs=-1, random_state=random_state, verbose=-1
            )
        elif name == "xgb":
            xgb = _get_xgb()
            return xgb.XGBClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                tree_method="hist", objective="binary:logistic",
                eval_metric="logloss", n_jobs=-1, random_state=random_state
            )
        else:
            raise ValueError("CLASSIFIER must be 'rf', 'gbt', 'lgb', or 'xgb'")

    def create_custom_colorbar(self, palette, layer_name, position_offset=0):
        """Create a small, custom-positioned colorbar with white background"""
        bottom_offset = 20 + (position_offset * 80)
        gradient_colors = ', '.join(palette)

        colorbar_html = f"""
        <div style='position: absolute; bottom: {bottom_offset}px; right: 20px; z-index: 9999;
                    background-color: rgba(255, 255, 255, 0.5); border: 1px solid #ccc;
                    border-radius: 4px; padding: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.2);
                    font-family: Arial, sans-serif;'>
            <div style='text-align: center; font-size: 10px; font-weight: bold; color: #333;
                        margin-bottom: 3px; white-space: nowrap;'>
                {layer_name}
            </div>
            <div style='width: 120px; height: 12px; background: linear-gradient(to right, {gradient_colors});
                        border: 1px solid #999; border-radius: 2px; margin-bottom: 2px;'></div>
            <div style='display: flex; justify-content: space-between; font-size: 8px; color: #666;
                        width: 120px;'>
                <span>0.0</span>
                <span>0.5</span>
                <span>1.0</span>
            </div>
        </div>
        """
        return colorbar_html

    def _update_history_table(self):
        """Refresh the history output widget with all runs so far"""
        with self.history_output:
            clear_output(wait=True)
            if not self.run_history:
                return
            df = pd.DataFrame(self.run_history)
            display(HTML(df.to_html(index=False, classes='table', border=0)))


    def _display_comparison_table(self, rows, years, algos, class_b_values=None):
        """Display styled comparison table with Class A/B, best-row highlight, CSV export, charts."""
        import matplotlib.pyplot as plt
        multi_year = len(years) > 1
        multi_algo = len(algos) > 1
        multi_cb   = len(class_b_values) > 1 if class_b_values else False
        algo_labels = {'rf': 'Random Forest', 'gbt': 'Gradient Boosting',
                       'xgb': 'XGBoost', 'lgb': 'LightGBM'}
        ca_name_t = self.ESA_CLASSES.get(self.class_a.value, str(self.class_a.value))
        cb_names_t = [self.ESA_CLASSES.get(v, str(v)) for v in (class_b_values or [])]
        classes_str = ca_name_t + ' vs ' + ', '.join(cb_names_t) if cb_names_t else ca_name_t
        _yr_f = self.year_from_dropdown.value
        _yr_t = self.year_to_dropdown.value
        year_range = str(_yr_f) + '-' + str(_yr_t) if _yr_f != _yr_t else str(_yr_f)
        algo_str = 'All Algorithms' if len(algos) == 4 else algo_labels.get(algos[0], algos[0])
        title = classes_str + ' | ' + year_range + ' | ' + algo_str
        base_cols = ['Class A', 'Class B']
        if multi_year and not multi_algo and not multi_cb:
            extra = ['Year']
        elif multi_algo and not multi_year and not multi_cb:
            extra = ['Algorithm']
        elif not multi_year and not multi_algo and multi_cb:
            extra = []
        else:
            extra = []
            if multi_year or (multi_cb and multi_year): extra.append('Year')
            if multi_algo: extra.append('Algorithm')
            if not extra:
                extra = ['Year', 'Algorithm']
        metric_cols = ['Accuracy', 'ROC AUC', 'Prec A', 'Rec A', 'F1 A',
                       'Prec B', 'Rec B', 'F1 B', 'Top Embedding', 'Best']
        cols = base_cols + extra + metric_cols
        th_s = 'padding:8px 12px;background:#1e40af;color:white;text-align:center;white-space:nowrap;'
        header_cells = ''.join('<th style="' + th_s + '">' + c + '</th>' for c in cols)
        def fmt_val(c, v):
            if c in ('Accuracy', 'Prec A', 'Rec A', 'F1 A', 'Prec B', 'Rec B', 'F1 B'):
                return str(round(float(v) * 100, 1)) + '%'
            if c == 'ROC AUC':
                return str(round(float(v), 3))
            return str(v)
        accs = [r['Accuracy'] for r in rows]
        best_idx = accs.index(max(accs)) if accs else -1
        body = ''
        td_s = 'padding:8px 12px;text-align:center;border-bottom:1px solid #e2e8f0;'
        td_b = 'padding:8px 12px;text-align:center;border-bottom:1px solid #e2e8f0;color:#166534;font-weight:bold;'
        for ridx, row in enumerate(rows):
            is_best = (ridx == best_idx)
            bg = '#d1fae5' if is_best else ('white' if ridx % 2 == 0 else '#f8fafc')
            row_ext = dict(row)
            row_ext['Best'] = '★ Best' if is_best else ''
            cells = ''
            for c in cols:
                s = td_b if (c == 'Best' and is_best) else td_s
                cells += '<td style="' + s + '">' + fmt_val(c, row_ext.get(c, '')) + '</td>'
            body += '<tr style="background:' + bg + ';">' + cells + '</tr>'
        tbl_s = 'border-collapse:collapse;width:100%;font-size:13px;font-family:Arial,sans-serif;'
        html = (
            '<div style="margin:20px 0;">'
            + '<h4 style="color:#1e40af;margin:0 0 10px 0;">' + title + '</h4>'
            + '<div style="overflow-x:auto;">'
            + '<table style="' + tbl_s + '">'
            + '<thead><tr>' + header_cells + '</tr></thead>'
            + '<tbody>' + body + '</tbody>'
            + '</table></div>'
            + '<p style="font-size:12px;color:#64748b;margin:6px 0 0;">★ Best row = highest accuracy.</p>'
            + '</div>'
        )
        display(HTML(html))
        csv_cols = [c for c in cols if c != 'Best']
        import io as _io
        _cbuf = _io.StringIO()
        pd.DataFrame([{c: row.get(c, '') for c in csv_cols} for row in rows],
                     columns=csv_cols).to_csv(_cbuf, index=False)
        _b64 = base64.b64encode(_cbuf.getvalue().encode()).decode()
        dl_link = (
            '<a href="data:text/csv;base64,' + _b64
            + '" download="comparison_table.csv"'
            + ' style="display:inline-block;padding:8px 16px;background:#0ea5e9;color:white;'
            + 'border-radius:6px;text-decoration:none;font-weight:bold;margin:10px 0;">'
            + 'Download Comparison Table as CSV</a>'
        )
        display(HTML(dl_link))
        if len(rows) > 1:
            import io as _io_charts
            chart_years = sorted(set(r['Year'] for r in rows))
            chart_algos = sorted(set(r['Algorithm'] for r in rows))
            clr = {'Random Forest': '#3b82f6', 'Gradient Boosting': '#f59e0b',
                   'XGBoost': '#10b981', 'LightGBM': '#8b5cf6'}
            metrics = [('Accuracy', 'Accuracy (%)'), ('ROC AUC', 'ROC AUC')]
            _dl_style = ('display:inline-block;padding:6px 14px;background:#6366f1;color:white;'
                         'border-radius:6px;text-decoration:none;font-weight:bold;margin:4px 0;font-size:12px;')
            if len(chart_years) > 1 and len(chart_algos) > 1:
                fig, axes = plt.subplots(1, 2, figsize=(14, 5))
                for mi, (mk, ml) in enumerate(metrics):
                    ax = axes[mi]
                    x = np.arange(len(chart_years))
                    w = 0.8 / max(len(chart_algos), 1)
                    for ai, alg in enumerate(chart_algos):
                        vals = []
                        for yr in chart_years:
                            m = [r for r in rows if r['Year'] == yr and r['Algorithm'] == alg]
                            v = float(m[0][mk]) if m else 0.0
                            vals.append(v * 100 if mk == 'Accuracy' else v)
                        offset = (ai - len(chart_algos) / 2 + 0.5) * w
                        ax.bar(x + offset, vals, width=w, label=alg, color=clr.get(alg))
                    ax.set_xticks(x)
                    ax.set_xticklabels([str(y) for y in chart_years])
                    ax.set_xlabel('Year')
                    ax.set_ylabel(ml)
                    ax.set_title(ml)
                    ax.legend(fontsize=8)
                plt.tight_layout()
                _b1 = _io_charts.BytesIO()
                fig.savefig(_b1, format='png', dpi=300, bbox_inches='tight')
                _b1.seek(0)
                _b64_1 = base64.b64encode(_b1.read()).decode()
                plt.show()
                display(HTML('<a href="data:image/png;base64,' + _b64_1 + '" download="accuracy_roc_chart.png" style="' + _dl_style + '">Download Chart as PNG</a>'))
            elif len(chart_years) > 1:
                fig, axes = plt.subplots(1, 2, figsize=(12, 4))
                for mi, (mk, ml) in enumerate(metrics):
                    vals = [float([r for r in rows if r['Year'] == yr][0][mk]) if [r for r in rows if r['Year'] == yr] else 0.0 for yr in chart_years]
                    vals = [v * 100 if mk == 'Accuracy' else v for v in vals]
                    axes[mi].bar([str(y) for y in chart_years], vals, color='#3b82f6')
                    axes[mi].set_xlabel('Year')
                    axes[mi].set_ylabel(ml)
                    axes[mi].set_title(ml)
                plt.tight_layout()
                _b2 = _io_charts.BytesIO()
                fig.savefig(_b2, format='png', dpi=300, bbox_inches='tight')
                _b2.seek(0)
                _b64_2 = base64.b64encode(_b2.read()).decode()
                plt.show()
                display(HTML('<a href="data:image/png;base64,' + _b64_2 + '" download="accuracy_roc_chart.png" style="' + _dl_style + '">Download Chart as PNG</a>'))
            else:
                fig, axes = plt.subplots(1, 2, figsize=(12, 4))
                for mi, (mk, ml) in enumerate(metrics):
                    vals = [float([r for r in rows if r['Algorithm'] == alg][0][mk]) if [r for r in rows if r['Algorithm'] == alg] else 0.0 for alg in chart_algos]
                    vals = [v * 100 if mk == 'Accuracy' else v for v in vals]
                    axes[mi].bar(chart_algos, vals, color=[clr.get(a, '#3b82f6') for a in chart_algos])
                    axes[mi].set_xlabel('Algorithm')
                    axes[mi].set_ylabel(ml)
                    axes[mi].set_title(ml)
                    axes[mi].tick_params(axis='x', rotation=20)
                plt.tight_layout()
                _b3 = _io_charts.BytesIO()
                fig.savefig(_b3, format='png', dpi=300, bbox_inches='tight')
                _b3.seek(0)
                _b64_3 = base64.b64encode(_b3.read()).decode()
                plt.show()
                display(HTML('<a href="data:image/png;base64,' + _b64_3 + '" download="accuracy_roc_chart.png" style="' + _dl_style + '">Download Chart as PNG</a>'))
            if len(chart_years) > 1:
                _year_embed = {}
                for _yr in chart_years:
                    _yr_rows = [r for r in rows if r.get('Year') == _yr and 'all_importances' in r]
                    if not _yr_rows:
                        continue
                    _embed_sums = {}
                    for _r in _yr_rows:
                        for _bname, _bval in _r['all_importances']:
                            _embed_sums[_bname] = _embed_sums.get(_bname, 0.0) + float(_bval)
                    _year_embed[_yr] = {k: v / len(_yr_rows) for k, v in _embed_sums.items()}
                _all_embed_avg = {}
                for _yd in _year_embed.values():
                    for k, v in _yd.items():
                        _all_embed_avg[k] = _all_embed_avg.get(k, 0.0) + v
                _top_embeds = sorted(_all_embed_avg, key=lambda k: _all_embed_avg[k], reverse=True)[:5]
                if _top_embeds and len(_year_embed) > 1:
                    _palette_t = ['#3b82f6', '#ef4444', '#10b981', '#f59e0b', '#8b5cf6']
                    _markers_t = ['o', 's', '^', 'D', 'v']
                    fig_t, ax_t = plt.subplots(figsize=(10, 5))
                    for _ei, _emb in enumerate(_top_embeds):
                        _yvals = [_year_embed.get(_yr, {}).get(_emb, None) for _yr in chart_years]
                        ax_t.plot(
                            [str(_yr) for _yr in chart_years], _yvals,
                            color=_palette_t[_ei % len(_palette_t)],
                            marker=_markers_t[_ei % len(_markers_t)],
                            linewidth=2, markersize=7, label=_emb
                        )
                    ax_t.set_xlabel('Year', fontsize=12)
                    ax_t.set_ylabel('Importance (%)', fontsize=12)
                    ax_t.set_title('Top 5 Embedding Importance Across Years', fontsize=14, fontweight='bold')
                    ax_t.legend(fontsize=9, loc='best', framealpha=0.9)
                    ax_t.grid(True, alpha=0.25, linestyle='--')
                    ax_t.spines['top'].set_visible(False)
                    ax_t.spines['right'].set_visible(False)
                    plt.tight_layout()
                    _bt = _io_charts.BytesIO()
                    fig_t.savefig(_bt, format='png', dpi=300, bbox_inches='tight')
                    _bt.seek(0)
                    _b64_t = base64.b64encode(_bt.read()).decode()
                    plt.show()
                    display(HTML('<a href="data:image/png;base64,' + _b64_t + '" download="embedding_temporal_chart.png" style="' + _dl_style + '">Download Chart as PNG</a>'))


    def run_analysis(self, button):
        """Main analysis function"""
        import matplotlib.pyplot as plt
        import folium
        with self.output_area:
            clear_output(wait=True)

            if self.country_dropdown.value == "Select your current country":
                self.status.value = """
                <div style='background: #fee2e2; padding: 12px; border-radius: 8px; margin: 10px 0;
                            border-left: 4px solid #ef4444; text-align: center;'>
                    <span style='color: #dc2626; font-weight: bold; font-size: 16px;'>
                        Please select the country you are currently in!
                    </span>
                </div>
                """
                return

            if self.selected_bbox is None:
                self.status.value = """
                <div style='background: #fee2e2; padding: 12px; border-radius: 8px; margin: 10px 0;
                            border-left: 4px solid #ef4444; text-align: center;'>
                    <span style='color: #dc2626; font-weight: bold; font-size: 16px;'>
                        Please draw a region on the map first!
                    </span>
                </div>
                """
                return

            if not EE_INITIALIZED:
                self.status.value = """
                <div style='background: #fef3c7; padding: 12px; border-radius: 8px; margin: 10px 0;
                            border-left: 4px solid #f59e0b; text-align: center;'>
                    <span style='color: #92400e; font-weight: bold; font-size: 16px;'>
                        Demo Mode Active - Earth Engine Not Available
                    </span>
                </div>
                """
                return

            if self.class_a.value == 999 and list(self.class_b.value) == [999]:
                self.progress.layout.visibility = 'hidden'
                self.status.value = """
                <div style='background: #fee2e2; padding: 12px; border-radius: 8px; margin: 10px 0;
                            border-left: 4px solid #ef4444; text-align: center;'>
                    <span style='color: #dc2626; font-weight: bold; font-size: 16px;'>
                        Both classes cannot be "All other classes"! Select one specific class.
                    </span>
                </div>
                """
                return

            if all(v == self.class_a.value for v in self.class_b.value):
                self.progress.layout.visibility = 'hidden'
                self.status.value = """
                <div style='background: #fee2e2; padding: 12px; border-radius: 8px; margin: 10px 0;
                            border-left: 4px solid #ef4444; text-align: center;'>
                    <span style='color: #dc2626; font-weight: bold; font-size: 16px;'>
                        Please select different classes!
                    </span>
                </div>
                """
                return

            self.progress.layout.visibility = 'visible'
            self.progress.value = 10
            self.progress.description = 'Starting...'

            self.status.value = """
            <div style='background: #dbeafe; padding: 12px; border-radius: 8px; margin: 10px 0;
                        border-left: 4px solid #3b82f6; text-align: center;'>
                <span style='color: #1e40af; font-weight: bold; font-size: 16px;'>
                    Processing embeddings and generating results...
                </span>
            </div>
            """

            try:
                # Determine years and algorithms
                years = list(range(self.year_from_dropdown.value, self.year_to_dropdown.value + 1))
                algo_labels = {'rf': 'Random Forest', 'gbt': 'Gradient Boosting',
                               'xgb': 'XGBoost', 'lgb': 'LightGBM'}
                algos = list(algo_labels.keys()) if self.algorithm.value == 'all' else [self.algorithm.value]
                class_b_values = list(self.class_b.value) if self.class_b.value else []
                if not class_b_values:
                    print('Please select at least one Class B.')
                    self.progress.layout.visibility = 'hidden'
                    return
                multi_mode = (len(class_b_values) > 1) or (len(years) > 1) or (len(algos) > 1)
                n_show = self.download_n_dropdown.value

                bbox = self.selected_bbox
                roi_info = sanitize_bbox(bbox)
                region = region_from_parts(roi_info["parts"])
                center_lat, center_lon = roi_info["center"]

                self.progress.value = 30
                self.progress.description = 'Loading satellite data...'

                self._cache_msg_printed = False
                comparison_rows = []
                ts_year_data = {}
                primary = None
                total_combos = len(class_b_values) * len(years) * len(algos)
                combo_idx = 0

                for class_b in class_b_values:
                    if class_b == self.class_a.value:
                        print('Skipping Class B=' + str(class_b) + ' (same as Class A). Selecting different classes is required.')
                        continue
                    for year in years:
                        df, EMBED_BANDS, hist, size, pair_fc = self.ee_pairwise_sample_with_others(
                            self.class_a.value, class_b,
                            year=year,
                            region=region,
                            scale_m=self.scale_m.value,
                            n_per_class=self.n_samples.value,
                            seed=self.seed.value
                        )

                        counts = df['label'].value_counts().to_dict() if len(df) else {}
                        have_a = counts.get(self.class_a.value, 0)
                        have_b = counts.get(class_b, 0)
                        MIN_PER_CLASS_REQUIRED = max(20, int(0.2 * self.n_samples.value))

                        if have_a < MIN_PER_CLASS_REQUIRED or have_b < MIN_PER_CLASS_REQUIRED:
                            _ca_warn = self.ESA_CLASSES.get(self.class_a.value, str(self.class_a.value))
                            _cb_warn = self.ESA_CLASSES.get(class_b, str(class_b))
                            _short_parts = []
                            if have_a < MIN_PER_CLASS_REQUIRED:
                                _short_parts.append(str(have_a) + ' for ' + _ca_warn)
                            if have_b < MIN_PER_CLASS_REQUIRED:
                                _short_parts.append(str(have_b) + ' for ' + _cb_warn)
                            _short_str = 'only ' + ' and '.join(_short_parts) + ' pixels found'
                            display(HTML(
                                "<div style='background:#fef9c3;border-left:4px solid #eab308;"
                                "padding:10px 14px;border-radius:6px;margin:6px 0;"
                                "font-size:13px;color:#713f12;'>"
                                + '⚠️ <b>' + _ca_warn + ' vs ' + _cb_warn
                                + ' skipped for ' + str(year) + '</b>'
                                + ' - ' + _short_str + ' in this area'
                                + ' (minimum ' + str(MIN_PER_CLASS_REQUIRED) + ' needed).'
                                + ' Try enlarging the ROI or reducing the Samples/class slider.'
                                + "</div>"
                            ))
                            continue

                        if class_b == class_b_values[0]:
                            ts_year_data[year] = (df, pair_fc)

                        label_map = {self.class_a.value: 0, class_b: 1}
                        df_y = df[df['label'].isin(label_map.keys())].copy()
                        y_arr = df_y['label'].map(label_map).astype(int).values
                        X_arr = df_y[EMBED_BANDS].astype(float).values

                        X_train, X_test, y_train, y_test = train_test_split(
                            X_arr, y_arr, test_size=self.test_size.value/100,
                            random_state=self.seed.value, stratify=y_arr
                        )

                        ca_name = self.ESA_CLASSES.get(self.class_a.value, str(self.class_a.value))
                        cb_name = self.ESA_CLASSES.get(class_b, str(class_b))

                        for alg in algos:
                            combo_idx += 1
                            pct = int(30 + 60 * combo_idx / total_combos)
                            self.progress.value = pct
                            self.progress.description = 'Training ' + algo_labels[alg] + ' (' + str(year) + ')...'

                            clf = self.build_model(alg, random_state=self.seed.value)
                            clf.fit(X_train, y_train)
                            y_proba = clf.predict_proba(X_test)[:, 1]
                            y_pred  = (y_proba >= 0.5).astype(int)

                            acc  = accuracy_score(y_test, y_pred)
                            auc  = roc_auc_score(y_test, y_proba)
                            report = classification_report(y_test, y_pred, output_dict=True)
                            cm_matrix = confusion_matrix(y_test, y_pred)
                            importances = clf.feature_importances_
                            top5 = top_k_importance(importances, EMBED_BANDS, k=n_show, as_percent=True)
                            all_importances = all_embedding_importance(importances, EMBED_BANDS, as_percent=True)
                            top1_name = top5[0][0] if top5 else ''
                            top1_pct  = top5[0][1] if top5 else 0.0


                            comparison_rows.append({
                                'Class A': ca_name,
                                'Class B': cb_name,
                                'Year': year,
                                'Algorithm': algo_labels[alg],
                                'Accuracy': acc,
                                'ROC AUC': auc,
                                'Prec A': report['0']['precision'],
                                'Rec A':  report['0']['recall'],
                                'F1 A':   report['0']['f1-score'],
                                'Prec B': report['1']['precision'],
                                'Rec B':  report['1']['recall'],
                                'F1 B':   report['1']['f1-score'],
                                'Top Embedding': top1_name + ' (' + f'{top1_pct:.1f}' + '%)',
                                'all_importances': all_importances,
                            })

                            if primary is None:
                                primary = dict(
                                    acc=acc, auc=auc, report=report,
                                    cm_matrix=cm_matrix, y_train=y_train, y_test=y_test,
                                    importances=importances, EMBED_BANDS=EMBED_BANDS,
                                    top5=top5, all_importances=all_importances,
                                    year=year, alg=alg, pair_fc=pair_fc,
                                    class_b=class_b,
                                )

                if not comparison_rows:
                    print('No results computed. Try enlarging the region or changing parameters.')
                    self.progress.layout.visibility = 'hidden'
                    return

                self.progress.value = 90
                self.progress.description = 'Creating map...'

                year         = primary['year']
                top5         = primary['top5']
                importances  = primary['importances']
                EMBED_BANDS  = primary['EMBED_BANDS']
                all_importances = primary['all_importances']
                acc          = primary['acc']
                auc          = primary['auc']
                report       = primary['report']
                cm_matrix    = primary['cm_matrix']
                y_train      = primary['y_train']
                y_test       = primary['y_test']
                pair_fc      = primary['pair_fc']

                emb_full, _, emb_mask_vis, esa_vis = load_stack(year)

                embedding_layers = []
                for i, (display_name, importance_pct, original_band) in enumerate(top5):
                    embedding_band = emb_full.select(original_band).updateMask(emb_mask_vis).clip(region)
                    normalized_embedding = normalize_embedding(embedding_band, region)
                    embedding_layers.append({
                        'image': normalized_embedding,
                        'name': display_name,
                        'importance': importance_pct,
                        'palette_key': 'embedding_' + str(i+1),
                        'rank': i + 1
                    })

                if self.class_a.value == 999 or primary['class_b'] == 999:
                    all_esa_classes = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
                    if self.class_a.value == 999:
                        specific_class = primary['class_b']
                        other_classes = [c for c in all_esa_classes if c != specific_class]
                        others_mask_vis = esa_vis.eq(other_classes[0])
                        for other_class in other_classes[1:]:
                            others_mask_vis = others_mask_vis.Or(esa_vis.eq(other_class))
                        relabeled_esa_vis = esa_vis.where(others_mask_vis, 999)
                    else:
                        specific_class = self.class_a.value
                        other_classes = [c for c in all_esa_classes if c != specific_class]
                        others_mask_vis = esa_vis.eq(other_classes[0])
                        for other_class in other_classes[1:]:
                            others_mask_vis = others_mask_vis.Or(esa_vis.eq(other_class))
                        relabeled_esa_vis = esa_vis.where(others_mask_vis, 999)
                    classA_img = relabeled_esa_vis.eq(self.class_a.value).updateMask(relabeled_esa_vis.eq(self.class_a.value)).updateMask(emb_mask_vis).clip(region)
                    classB_img = relabeled_esa_vis.eq(primary['class_b']).updateMask(relabeled_esa_vis.eq(primary['class_b'])).updateMask(emb_mask_vis).clip(region)
                else:
                    classA_img = esa_vis.eq(self.class_a.value).updateMask(esa_vis.eq(self.class_a.value)).updateMask(emb_mask_vis).clip(region)
                    classB_img = esa_vis.eq(primary['class_b']).updateMask(esa_vis.eq(primary['class_b'])).updateMask(emb_mask_vis).clip(region)

                m = folium.Map(location=[center_lat, center_lon], zoom_start=8, control_scale=True)
                folium.TileLayer(
                    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
                    attr='Google', name='Google Satellite', overlay=False, control=True
                ).add_to(m)
                color_a = self.CLASS_COLORS.get(self.class_a.value, '#FF3B30')
                color_b = self.CLASS_COLORS.get(primary['class_b'], '#007AFF')
                add_ee_layer(m, classA_img, {'palette': [color_a]},
                             'Pixels: ' + str(self.class_a.value) + ' (' + self.ESA_CLASSES.get(self.class_a.value, '?') + ')')
                add_ee_layer(m, classB_img, {'palette': [color_b]},
                             'Pixels: ' + str(primary['class_b']) + ' (' + self.ESA_CLASSES.get(primary['class_b'], '?') + ')')

                all_colorbars_html = ''
                for i, layer in enumerate(embedding_layers):
                    try:
                        palette_key = layer['palette_key']
                        if palette_key not in EMBEDDING_PALETTES:
                            palette_key = 'embedding_' + str((i % 10) + 1)
                        palette = EMBEDDING_PALETTES[palette_key]
                        layer_name = layer['name'] + ' (' + f"{layer['importance']:.1f}" + '%)'
                        add_ee_layer(m, layer['image'], {'min': 0, 'max': 1, 'palette': palette}, layer_name)
                        all_colorbars_html += self.create_custom_colorbar(palette, layer['name'], position_offset=i)
                    except Exception as layer_err:
                        print('  Failed to add layer ' + layer['name'] + ': ' + str(layer_err))
                        continue

                if all_colorbars_html:
                    m.get_root().html.add_child(folium.Element(all_colorbars_html))

                sample_points = pair_fc.getInfo()
                if 'features' in sample_points:
                    class_a_name = self.ESA_CLASSES.get(self.class_a.value, '?')
                    class_b_name = self.ESA_CLASSES.get(primary['class_b'], '?')
                    points_layer_a = folium.FeatureGroup(name='Samples: ' + str(self.class_a.value) + ' (' + class_a_name + ')')
                    points_layer_b = folium.FeatureGroup(name='Samples: ' + str(primary['class_b']) + ' (' + class_b_name + ')')
                    for feature in sample_points['features']:
                        coords = feature['geometry']['coordinates']
                        label  = feature['properties']['label']
                        if label == self.class_a.value:
                            folium.CircleMarker(
                                location=[coords[1], coords[0]], radius=4,
                                popup='Class ' + str(label) + ' (' + class_a_name + ')',
                                color='white', fillColor='black', fillOpacity=1, weight=1
                            ).add_to(points_layer_a)
                        elif label == primary['class_b']:
                            folium.CircleMarker(
                                location=[coords[1], coords[0]], radius=4,
                                popup='Class ' + str(label) + ' (' + class_b_name + ')',
                                color='black', fillColor='white', fillOpacity=1, weight=1
                            ).add_to(points_layer_b)
                    points_layer_a.add_to(m)
                    points_layer_b.add_to(m)

                if self.upload_geojson is not None:
                    roi_layer = folium.FeatureGroup(name='ROI Boundary')
                    folium.GeoJson(
                        self.upload_geojson,
                        style_function=lambda x: {'color': 'black', 'weight': 2, 'fillOpacity': 0}
                    ).add_to(roi_layer)
                    roi_layer.add_to(m)
                else:
                    roi_coords = []
                    for part in roi_info['parts']:
                        minLon, minLat, maxLon, maxLat = part
                        roi_coords.append([
                            [minLat, minLon], [minLat, maxLon],
                            [maxLat, maxLon], [maxLat, minLon], [minLat, minLon]
                        ])
                    roi_layer = folium.FeatureGroup(name='ROI Boundary')
                    for coords in roi_coords:
                        folium.Polygon(
                            locations=coords, color='black', weight=2, fill=False,
                            popup='Selected ROI Boundary'
                        ).add_to(roi_layer)
                    roi_layer.add_to(m)
                folium.LayerControl().add_to(m)

                if multi_mode:
                    self._display_comparison_table(comparison_rows, years, algos, class_b_values)
                    if len(class_b_values) > 1:
                        _ca_n = self.ESA_CLASSES.get(self.class_a.value, str(self.class_a.value))
                        for _cb_val in class_b_values:
                            _cb_n = self.ESA_CLASSES.get(_cb_val, str(_cb_val))
                            _cb_rows = [r for r in comparison_rows if r.get('Class B') == _cb_n]
                            if not _cb_rows:
                                continue
                            _best = max(_cb_rows, key=lambda r: r.get('Accuracy', 0))
                            _all_imp = _best.get('all_importances', [])
                            if not _all_imp:
                                continue
                            _alg = _best.get('Algorithm', self.algorithm.value.upper())
                            _arr = np.array([v for _, v in _all_imp])
                            _xlabels = [n for n, _ in _all_imp]
                            _top_idx = set(np.argsort(_arr)[-n_show:])
                            _colors = ['yellow' if i in _top_idx else 'steelblue'
                                       for i in range(len(_arr))]
                            plt.figure(figsize=(14, 4))
                            plt.bar(range(len(_arr)), _arr, color=_colors)
                            plt.xticks(range(len(_arr)), _xlabels, rotation=90)
                            plt.ylabel('Importance (%)')
                            plt.title(
                                'Embedding importance: ' + _ca_n + ' vs ' + _cb_n
                                + ' [model = ' + _alg + ']'
                                + ' (Yellow = Top ' + str(n_show) + ' highest)'
                            )
                            plt.tight_layout()
                            plt.show()

                if len(class_b_values) == 1:
                    self.display_metrics(acc, auc, report, cm_matrix, len(y_train), len(y_test),
                                         importances, EMBED_BANDS, top5, all_importances,
                                         class_b=primary['class_b'], multi_mode=multi_mode,
                                         primary_year=primary['year'], primary_alg=primary['alg'],
                                         n_combos=len(comparison_rows))

                print('Use the layer control panel to toggle layers on/off')
                print('Color bars in bottom-right show normalized embedding values (0.0-1.0)')
                display(m)

                download_button = widgets.Button(
                    description='Download Top ' + str(n_show) + ' Embedding Rasters',
                    button_style='info',
                    layout=widgets.Layout(width='400px', height='45px', margin='15px 0'),
                    tooltip='Generate download links for the top ' + str(n_show) + ' most important embeddings'
                )
                download_status = widgets.HTML()

                def download_embeddings(b):
                    download_status.value = """
                    <div style='background: #dbeafe; padding: 10px; border-radius: 8px; margin: 10px 0; text-align: center;'>
                        <strong>Generating download links...</strong> This may take a few moments.
                    </div>
                    """
                    try:
                        download_links_html = "<div style='background: #f0f9ff; padding: 15px; border-radius: 8px; margin: 10px 0;'>"
                        download_links_html += """
                        <h4 style='color: #1e40af; margin: 0 0 10px 0; text-align: center;'>Download Instructions:</h4>
                        <div style='background: #fff3cd; padding: 10px; border-radius: 6px; margin-bottom: 10px; font-size: 13px; color: #92400e;'>
                            <strong>Important:</strong> Earth Engine gives complicated filenames. Copy the suggested name below.
                        </div>
                        """
                        for i, layer in enumerate(embedding_layers):
                            print('Generating download link for ' + layer['name'] + ' (Rank ' + str(layer['rank']) + ')...')
                            clean_filename = layer['name'] + '_' + f"{bbox[0]:.2f}" + '_' + f"{bbox[2]:.2f}" + '_' + f"{bbox[1]:.2f}" + '_' + f"{bbox[3]:.2f}" + '.tif'
                            download_url = layer['image'].getDownloadURL({
                                'scale': self.scale_m.value, 'crs': 'EPSG:4326',
                                'region': region, 'format': 'GEO_TIFF'
                            })
                            download_links_html += (
                                "<div style='margin: 8px 0; padding: 12px; background: white; border-radius: 6px; border-left: 3px solid #3b82f6;'>"
                                + '<strong>' + layer['name'] + '</strong> (' + f"{layer['importance']:.1f}" + '% importance)<br>'
                                + "<div style='background: #f1f5f9; padding: 8px; border-radius: 4px; margin: 6px 0;'>"
                                + '<strong>Suggested filename:</strong><br>'
                                + '<code>' + clean_filename + '</code>'
                                + '</div>'
                                + '<a href="' + download_url + '" target="_blank" style="color: #2563eb; font-weight: 500;">'
                                + 'Click &#8594; Save As &#8594; Use filename above</a>'
                                + '</div>'
                            )
                        download_links_html += '</div>'
                        download_status.value = (
                            "<div style='background: #dcfce7; padding: 10px; border-radius: 8px; margin: 10px 0; text-align: center;'>"
                            + '<strong>Download links ready!</strong></div>'
                            + download_links_html
                        )
                    except Exception as dl_err:
                        download_status.value = (
                            "<div style='background: #fee2e2; padding: 10px; border-radius: 8px; margin: 10px 0; text-align: center;'>"
                            + '<strong>Download generation failed:</strong> ' + str(dl_err) + '</div>'
                        )

                download_button.on_click(download_embeddings)
                display(widgets.VBox([download_button, download_status],
                                     layout=widgets.Layout(align_items='center')))

                png_dl_button = widgets.Button(
                    description='Download Top ' + str(n_show) + ' Embedding PNGs',
                    button_style='success',
                    layout=widgets.Layout(width='400px', height='45px', margin='15px 0'),
                    tooltip='Export each embedding as a coloured PNG with colorbar'
                )
                png_dl_status = widgets.HTML()

                def download_embedding_pngs(b):
                    png_dl_status.value = (
                        "<div style='background:#dbeafe;padding:10px;border-radius:8px;"
                        "margin:10px 0;text-align:center;'>"
                        "<strong>Generating PNG images...</strong> "
                        "Fetching thumbnails from Earth Engine."
                        "</div>"
                    )
                    try:
                        import urllib.request as _ureq
                        import io as _png_io
                        import numpy as _np_png
                        import matplotlib.pyplot as _plt_png
                        import matplotlib.image as _mimg
                        from matplotlib.colors import LinearSegmentedColormap as _LSCM
                        from matplotlib.cm import ScalarMappable as _SMappable
                        from matplotlib.colors import Normalize as _Norm
                        _dl_links = []
                        for _pi, _layer in enumerate(embedding_layers):
                            _pk = _layer['palette_key']
                            if _pk not in EMBEDDING_PALETTES:
                                _pk = 'embedding_' + str((_pi % 10) + 1)
                            _hex_pal = EMBEDDING_PALETTES[_pk]
                            _thumb_url = _layer['image'].getThumbURL({
                                'min': 0, 'max': 1,
                                'palette': _hex_pal,
                                'region': region,
                                'dimensions': 512,
                                'format': 'png'
                            })
                            with _ureq.urlopen(_thumb_url) as _resp:
                                _raw = _resp.read()
                            _img_arr = _mimg.imread(_png_io.BytesIO(_raw))
                            _rgb_list = [
                                tuple(int(_h.lstrip('#')[_j:_j+2], 16) / 255.0
                                      for _j in (0, 2, 4))
                                for _h in _hex_pal
                            ]
                            _cmap = _LSCM.from_list('emb_pal', _rgb_list, N=256)
                            _fig, _ax = _plt_png.subplots(figsize=(8, 7))
                            _ax.imshow(_img_arr)
                            _ax.axis('off')
                            _sm = _SMappable(cmap=_cmap, norm=_Norm(vmin=0, vmax=1))
                            _sm.set_array([])
                            _cbar = _fig.colorbar(_sm, ax=_ax, shrink=0.55, pad=0.03)
                            _cbar.set_label('Normalized Embedding Value (0 - 1)', fontsize=9)
                            _cbar.ax.tick_params(labelsize=8)
                            _ax.set_title(
                                _layer['name'] + '  |  '
                                + f"{_layer['importance']:.1f}% importance",
                                fontsize=13, fontweight='bold', pad=12
                            )
                            _plt_png.tight_layout()
                            _buf_out = _png_io.BytesIO()
                            _fig.savefig(_buf_out, format='png', dpi=150, bbox_inches='tight')
                            _buf_out.seek(0)
                            _b64_png = base64.b64encode(_buf_out.read()).decode()
                            _plt_png.close(_fig)
                            _fname = (
                                _layer['name'].replace(' ', '_')
                                + '_' + f"{_layer['importance']:.1f}pct.png"
                            )
                            _dl_links.append(
                                '<a href="data:image/png;base64,' + _b64_png
                                + '" download="' + _fname + '"'
                                + ' style="display:inline-block;padding:6px 14px;'
                                + 'background:#10b981;color:white;border-radius:6px;'
                                + 'text-decoration:none;font-weight:bold;margin:4px;font-size:12px;">'
                                + _layer['name']
                                + ' (' + f"{_layer['importance']:.1f}" + '%)</a>'
                            )
                        png_dl_status.value = (
                            "<div style='background:#dcfce7;padding:10px;border-radius:8px;"
                            "margin:10px 0;text-align:center;'>"
                            "<strong>PNGs ready — click each to download:</strong></div>"
                            + "<div style='margin:8px 0;text-align:center;'>"
                            + ' '.join(_dl_links) + '</div>'
                        )
                    except Exception as _png_err:
                        png_dl_status.value = (
                            "<div style='background:#fee2e2;padding:10px;border-radius:8px;"
                            "margin:10px 0;text-align:center;'>"
                            + '<strong>PNG export failed:</strong> ' + str(_png_err) + '</div>'
                        )

                png_dl_button.on_click(download_embedding_pngs)
                display(widgets.VBox([png_dl_button, png_dl_status],
                                     layout=widgets.Layout(align_items='center')))
                if len(years) > 1:
                    ts_btn = widgets.Button(
                        description='Download Time Series Data',
                        button_style='warning',
                        layout=widgets.Layout(width='400px', height='45px', margin='15px 0'),
                        tooltip='Export lat/lon/year/class/embeddings for all sampled pixels'
                    )
                    ts_status = widgets.HTML()

                    def download_timeseries(b):
                        ts_status.value = "<div style='background:#dbeafe;padding:10px;border-radius:8px;text-align:center;'><strong>Building time series CSV...</strong></div>"
                        try:
                            import io as _tsio
                            import datetime as _dt
                            t_res = self.temporal_res_dropdown.value
                            ts_rows = []
                            for _yr, (_df_yr, _pfc_yr) in sorted(ts_year_data.items()):
                                try:
                                    _feats = _pfc_yr.getInfo().get('features', [])
                                except Exception:
                                    _feats = []
                                _coords = {}
                                for _fi, _feat in enumerate(_feats):
                                    _geom = _feat.get('geometry', {})
                                    if _geom and _geom.get('type') == 'Point':
                                        _coords[_fi] = (_geom['coordinates'][1], _geom['coordinates'][0])
                                    else:
                                        _coords[_fi] = (None, None)
                                _ecols = sorted([c for c in _df_yr.columns if c != 'label'],
                                                key=lambda b: int(b[1:]) if b[1:].isdigit() else 0)
                                _crename = {}
                                for _b in _ecols:
                                    if _b.startswith('B') and _b[1:].isdigit():
                                        _crename[_b] = 'A' + str(int(_b[1:]) + 1).zfill(2)
                                    else:
                                        _crename[_b] = _b
                                for _ri, _row_r in _df_yr.reset_index(drop=True).iterrows():
                                    _lat, _lon = _coords.get(_ri, (None, None))
                                    _base = {'latitude': _lat, 'longitude': _lon, 'year': _yr,
                                             'land_cover_class': _row_r.get('label', '')}
                                    for _b in _ecols:
                                        _base[_crename.get(_b, _b)] = _row_r.get(_b, '')
                                    if t_res == 'Monthly':
                                        _mf = self.month_from_dropdown.value
                                        _mt = self.month_to_dropdown.value
                                        for _mon in range(_mf, _mt + 1):
                                            _r2 = dict(_base); _r2['month'] = _mon; ts_rows.append(_r2)
                                    elif t_res == 'Daily':
                                        try:
                                            _df_str = self.day_from_picker.value or str(_yr) + '-01-01'
                                            _dt_str = self.day_to_picker.value or str(_yr) + '-12-31'
                                            _d = _dt.date.fromisoformat(_df_str)
                                            _de = _dt.date.fromisoformat(_dt_str)
                                            while _d <= _de:
                                                _r2 = dict(_base); _r2['date'] = str(_d); ts_rows.append(_r2)
                                                _d += _dt.timedelta(days=1)
                                        except Exception:
                                            _r2 = dict(_base); _r2['date'] = str(_yr) + '-01-01'; ts_rows.append(_r2)
                                    else:
                                        ts_rows.append(_base)
                            _ts_df = pd.DataFrame(ts_rows)
                            _ts_buf = _tsio.StringIO()
                            _ts_df.to_csv(_ts_buf, index=False)
                            _ts_b64 = base64.b64encode(_ts_buf.getvalue().encode()).decode()
                            _ts_link = (
                                '<a href="data:text/csv;base64,' + _ts_b64
                                + '" download="timeseries_data.csv"'
                                + ' style="display:inline-block;padding:8px 16px;background:#f59e0b;color:white;'
                                + 'border-radius:6px;text-decoration:none;font-weight:bold;margin:10px 0;">'
                                + 'Download Time Series CSV (' + str(len(_ts_df)) + ' rows)</a>'
                            )
                            ts_status.value = _ts_link
                        except Exception as _ts_err:
                            ts_status.value = "<div style='background:#fee2e2;padding:10px;border-radius:8px;'>Error: " + str(_ts_err) + "</div>"

                    ts_btn.on_click(download_timeseries)
                    display(widgets.VBox([ts_btn, ts_status], layout=widgets.Layout(align_items='center')))

                self.feedback_section.layout.visibility = 'visible'
                self.status.value = """
                <div style='background: #dcfce7; padding: 12px; border-radius: 8px; margin: 10px 0;
                            border-left: 4px solid #16a34a; text-align: center;'>
                    <span style='color: #166534; font-weight: bold; font-size: 16px;'>
                        Analysis completed! Check results above and provide feedback below.
                    </span>
                </div>
                """
                self.progress.layout.visibility = 'hidden'

                results_data = {
                    'center_lat': center_lat, 'center_lon': center_lon,
                    'accuracy': acc, 'roc_auc': auc,
                    'precision_a': report['0']['precision'],
                    'recall_a':    report['0']['recall'],
                    'f1_a':        report['0']['f1-score'],
                    'precision_b': report['1']['precision'],
                    'recall_b':    report['1']['recall'],
                    'f1_b':        report['1']['f1-score'],
                    'confusion_matrix': cm_matrix,
                    'n_train': len(y_train), 'n_test': len(y_test),
                    'top_embeddings': [(name, float(pct)) for name, pct, _ in top5],
                    'all_embeddings': all_importances,
                    'class_b': primary['class_b'],
                }
                self.log_interaction_to_sheets(results_data)

                from datetime import datetime
                for row in comparison_rows:
                    self.run_history.append({
                        '#': len(self.run_history) + 1,
                        'Time': datetime.now().strftime('%H:%M:%S'),
                        'Year': row['Year'],
                        'Class A': self.ESA_CLASSES.get(self.class_a.value, str(self.class_a.value)),
                        'Class B': row['Class B'],
                        'Algorithm': row['Algorithm'],
                        'Accuracy': f"{row['Accuracy']*100:.1f}%",
                        'ROC AUC': f"{row['ROC AUC']:.3f}",
                        'Top Embedding': row['Top Embedding'],
                        'N Train': len(y_train),
                        'N Test': len(y_test),
                    })
                self._update_history_table()

            except Exception as e:
                print(f"Error: {str(e)}")
                self.progress.layout.visibility = 'hidden'
                self.status.value = f"""
                <div style='background: #fee2e2; padding: 12px; border-radius: 8px; margin: 10px 0;
                            border-left: 4px solid #ef4444; text-align: center;'>
                    <span style='color: #dc2626; font-weight: bold; font-size: 16px;'>
                        Error: {str(e)}
                    </span>
                </div>
                """

    def display_metrics(self, acc, auc, report, cm, n_train, n_test, importances, embed_bands, top5, all_importances=None, class_b=None, multi_mode=False, primary_year=None, primary_alg=None, n_combos=0):
        """Display classification results with CSV download for all 64 embedding importances"""
        import matplotlib.pyplot as plt
        metrics_html = f"""
        <div style='display: flex; flex-wrap: wrap; gap: 15px; margin: 20px 0;'>
            <div style='background: #dcfce7; padding: 15px; border-radius: 8px; text-align: center; min-width: 120px;'>
                <h4 style='margin: 0; color: #166534;'>Accuracy</h4>
                <h2 style='margin: 5px 0 0 0; color: #16a34a;'>{acc*100:.1f}%</h2>
            </div>
            <div style='background: #dbeafe; padding: 15px; border-radius: 8px; text-align: center; min-width: 120px;'>
                <h4 style='margin: 0; color: #1e40af;'>ROC AUC</h4>
                <h2 style='margin: 5px 0 0 0; color: #2563eb;'>{auc:.3f}</h2>
            </div>
            <div style='background: #f3e8ff; padding: 15px; border-radius: 8px; text-align: center; min-width: 120px;'>
                <h4 style='margin: 0; color: #7c2d12;'>Train</h4>
                <h2 style='margin: 5px 0 0 0; color: #a855f7;'>{n_train}</h2>
            </div>
            <div style='background: #fed7d7; padding: 15px; border-radius: 8px; text-align: center; min-width: 120px;'>
                <h4 style='margin: 0; color: #7c2d12;'>Test</h4>
                <h2 style='margin: 5px 0 0 0; color: #dc2626;'>{n_test}</h2>
            </div>
        </div>
        """

        display(HTML(metrics_html))

        print("\n=== Classification Results ===")
        print(f"Per-class metrics:")
        print(f"   Class {self.class_a.value}: Precision={report['0']['precision']:.3f}, Recall={report['0']['recall']:.3f}, F1={report['0']['f1-score']:.3f}")
        print(f"   Class {class_b if class_b is not None else (list(self.class_b.value)[0] if self.class_b.value else 0)}: Precision={report['1']['precision']:.3f}, Recall={report['1']['recall']:.3f}, F1={report['1']['f1-score']:.3f}")

        print("\nConfusion matrix (rows=true, cols=pred):")
        cm_df = pd.DataFrame(cm, index=['A(0)','B(1)'], columns=['P0','P1'])
        print(cm_df)

        n_show = len(top5)
        print(f"\nTop {n_show} Most Important Embeddings:")
        for i, (display_name, importance_pct, original_band) in enumerate(top5):
            print(f"   {i+1}. {display_name}: {importance_pct:.2f}%")

        # CSV download button for all 64 embedding importances
        if all_importances is not None:
            csv_buf = io.StringIO()
            pd.DataFrame(all_importances, columns=['Embedding', 'Importance (%)']
                        ).to_csv(csv_buf, index=False)
            b64 = base64.b64encode(csv_buf.getvalue().encode()).decode()
            csv_link = '<a href="data:text/csv;base64,' + b64 + '" download="embedding_importances.csv" style="display:inline-block;padding:8px 16px;background:#0ea5e9;color:white;border-radius:6px;text-decoration:none;font-weight:bold;margin:10px 0;">Download CSV</a>'
            display(HTML(csv_link))

        vals = np.asarray(importances, dtype=float).ravel()
        bands = list(embed_bands)
        L = min(vals.size, len(bands))
        vals, bands = vals[:L], bands[:L]
        idx = np.array([int(b[1:]) for b in bands])
        order = np.argsort(idx)
        arr = vals[order]
        total = arr.sum() or 1.0
        arr = 100.0 * arr / total
        xlabels = [f"A{n+1:02d}" for n in idx[order]]

        top_indices = set(np.argsort(importances)[-n_show:])
        colors = ['yellow' if i in top_indices else 'steelblue' for i in order]

        plt.figure(figsize=(14, 4))
        plt.bar(range(len(arr)), arr, color=colors)
        plt.xticks(range(len(arr)), xlabels, rotation=90)
        plt.ylabel("Importance (%)")

        class_a_name = self.ESA_CLASSES.get(self.class_a.value, f"Class {self.class_a.value}")
        class_b_name = self.ESA_CLASSES.get(class_b if class_b is not None else (list(self.class_b.value)[0] if self.class_b.value else 0), f"Class {class_b if class_b is not None else (list(self.class_b.value)[0] if self.class_b.value else 0)}")

        plt.title(f"Embedding importance: {class_a_name} vs {class_b_name} [model = {self.algorithm.value.upper()}] (Yellow = Top {n_show} highest)")
        plt.tight_layout()
        plt.show()
        if multi_mode:
            _note_ca = self.ESA_CLASSES.get(self.class_a.value, str(self.class_a.value))
            _note_cb = self.ESA_CLASSES.get(class_b, str(class_b)) if class_b is not None else '?'
            _alg_labels_n = {'rf': 'Random Forest', 'gbt': 'Gradient Boosting',
                             'xgb': 'XGBoost', 'lgb': 'LightGBM'}
            _alg_name_n = _alg_labels_n.get(primary_alg, str(primary_alg)) if primary_alg else self.algorithm.value
            _yr_n = str(primary_year) if primary_year is not None else '?'
            _combo_str = str(n_combos) + ' combination' + ('s' if n_combos != 1 else '')
            display(HTML(
                "<div style='background:#eff6ff;border-left:4px solid #3b82f6;padding:10px 14px;"
                "border-radius:6px;margin:12px 0;font-size:13px;color:#1e40af;'>"
                "ℹ️ <b>The metrics above show the last run only:</b> "
                + _note_ca + ' vs ' + _note_cb + ', ' + _yr_n + ', ' + _alg_name_n + '. '
                + 'The full comparison table above contains all ' + _combo_str + '.'
                + "</div>"
            ))

In [4]:
# ===============================================
# 🚀 LAUNCH THE APP
# ===============================================
app = AlphaEarthApp(PROJECT_ID)
app.display_app()    